In [1]:
import sys
print(sys.version)

import networkx as nx
print(nx.__version__)

3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
3.4.2


In [16]:
# CELL 1 — Load dataset and strip old STRING PPI edges
#
#   Load the existing Excel dataset that was built by the original KEGG, ChEMBL and STRING extraction pipeline. 
#   The KEGG and ChEMBL data are correct and do not need to be re-fetched. 
#   However, the STRING protein-protein interaction (PPI) edges were fetched at a confidence threshold of 0.7 (high confidence), 
#   which produced 1,179 edges and made the graph too dense to visualise clearly. 
#   This cell removes those edges so they can be re-fetched at 0.9 (very high confidence) in Cell 2.
#
# INPUT:  T2DM_Melanoma_KG_Dataset_auto5.xlsx (Sheet1 = nodes, Sheet2 = edges)
# OUTPUT: nodes (list of dicts), edges (list of dicts, STRING edges removed)


import pandas as pd       
import networkx as nx     

# ── Step 1: Load the Excel dataset ───────────────────────────────────────────
# Sheet1 contains nodes (diseases, pathways, genes, drugs, metabolites).
# Sheet2 contains edges (relationships between those nodes).
# Both sheets were produced by the original extraction pipeline and saved
# to Excel as the canonical dataset.

nodes_df = pd.read_excel(
    r"C:\Users\USER\OneDrive - Teesside University\T2DM_Melanoma_KG_Dataset_auto5.xlsx",
    sheet_name="Sheet1"   # node table: node_id, node_type, name, source_db, source_ref, notes
)
edges_df = pd.read_excel(
    r"C:\Users\USER\OneDrive - Teesside University\T2DM_Melanoma_KG_Dataset_auto5.xlsx",
    sheet_name="Sheet2"   # edge table: source_id, relation, target_id, source_db, source_ref, notes
)

print(f"Loaded: {len(nodes_df)} nodes, {len(edges_df)} edges")

# ── Step 2: Convert DataFrames to lists of dicts ──────────────────────────────
# The original pipeline stored nodes and edges as lists of dictionaries.
# We convert back to that format here so the rest of the code remains consistent with the original pipeline structure.

nodes = nodes_df.to_dict(orient="records")   # each row becomes one dict
edges = edges_df.to_dict(orient="records")   # each row becomes one dict

# ── Step 3: Remove all STRING INTERACTS_WITH edges ────────────────────────────
# STRING PPI edges have relation == "INTERACTS_WITH" and source_db == "STRING".
# We filter them out by relation type, which is the most reliable identifier.
# The 0.7 threshold produced 1,179 PPI edges — too many for clear visualisation.
# Re-fetching at 0.9 in Cell 2 below will keep only the most biologically reliable interactions, 
# reducing density while preserving the most meaningful connections.

before = len(edges)   # record edge count before removal for reporting

edges = [
    e for e in edges
    if e.get("relation") != "INTERACTS_WITH"   # keep everything that is NOT a STRING PPI edge
]

removed = before - len(edges)   # calculate how many STRING edges were removed

print(f"Removed {removed} STRING PPI edges (threshold was 0.7 — too dense)")
print(f"Remaining after removal: {len(nodes)} nodes, {len(edges)} edges")
print("Ready for Cell 2: STRING re-fetch at threshold 0.9")

Loaded: 290 nodes, 1567 edges
Removed 1179 STRING PPI edges (threshold was 0.7 — too dense)
Remaining after removal: 290 nodes, 388 edges
Ready for Cell 2: STRING re-fetch at threshold 0.9


In [17]:
# CELL 2 — Re-fetch STRING protein-protein interactions at threshold 0.9
#
# PURPOSE:
#   Query the STRING database API for protein-protein interactions (PPIs)
#   between all gene nodes currently in the dataset. The previous fetch used
#   a threshold of 0.7 (high confidence, STRING score ≥ 700/1000), which
#   produced 1,179 edges and made the graph too visually dense. This cell
#   re-fetches at 0.9 (very high confidence, STRING score ≥ 900/1000),
#   retaining only interactions with the strongest experimental and
#   computational evidence. This follows established practice in network
#   biology for reducing false positive edges while preserving the most
#   biologically reliable interactions.
#
# DATABASE: STRING v12.0 (https://string-db.org)
# SPECIES:  Homo sapiens (taxon ID 9606)
# THRESHOLD: 0.9 (very high confidence) = score_threshold=900 in STRING API
#
# IMPORTANT TO NOTE: The API calls to STRING are live, requiring an 
#   internet connection. STRING rate-limits large requests automatically,
#   so a 1-second pause is added between batches to be respectful of the API.
#
# INPUT:  nodes (list of dicts from Cell 1)
# OUTPUT: edges (list of dicts, STRING PPI edges added at threshold 0.9)


import requests   # for making HTTP requests to the STRING REST API
import time       # for adding pauses between API calls to avoid rate limiting

# ── Helper function: add_edge ─────────────────────────────────────────────────
# This replicates the add_edge helper from the original pipeline.
# It appends a new edge dictionary to the global edges list.
# Defined here again so this notebook is self-contained and does not
# depend on the original pipeline cells having been run first.

def add_edge(source_id, relation, target_id, source_db, source_ref, notes=""):
    edges.append({
        "source_id":  source_id,    # node_id of the node where the relationship starts
        "relation":   relation,     # type of relationship (e.g. "INTERACTS_WITH")
        "target_id":  target_id,    # node_id of the node where the relationship ends
        "source_db":  source_db,    # database that asserts this relationship
        "source_ref": source_ref,   # URL or accession pointing to the evidence
        "notes":      notes,        # optional free-text notes about this edge
    })


# ── Helper function: get_string_interactions ──────────────────────────────────
# Queries the STRING API for all PPIs between a given list of gene symbols.
# STRING accepts up to ~2000 identifiers per request. For safety, we batch
# into groups of 400, pausing between batches to respect rate limits.
# Returns a list of (gene_a, gene_b, combined_score) tuples where combined_score is a float between 0 and 1.

def get_string_interactions(gene_list, species=9606, score_threshold=900):
    """
    Fetch protein-protein interactions from STRING v12.0.
    species=9606 is Homo sapiens.
    score_threshold: integer 0-1000.
        400 = medium confidence
        700 = high confidence
        900 = very high confidence  ← we use this
    Returns a list of (gene_a, gene_b, score) tuples.
    """
    url = "https://string-db.org/api/json/network"   # STRING REST API endpoint

    # STRING asks callers to identify their application — good API practice
    params = {
        "identifiers":     "%0d".join(gene_list),    # genes joined by URL-encoded newline
        "species":         species,                  # 9606 = Homo sapiens
        "caller_identity": "t2dm_melanoma_kg",       # identifier for this project
        "required_score":  score_threshold,          # minimum combined confidence score
    }

    try:
        resp = requests.get(url, params=params, timeout=60)   # 60s timeout for large gene lists
        resp.raise_for_status()                               # raise error on HTTP 4xx/5xx
        data = resp.json()                                    # parse JSON response
    except Exception as e:
        print(f"  Warning: STRING API call failed — {e}")
        return []   # return empty list so the pipeline can continue gracefully

    # Parse response: each entry is one interaction with two protein names and a score
    interactions = []
    for entry in data:
        gene_a = entry.get("preferredName_A", "")   # symbol of first interacting protein
        gene_b = entry.get("preferredName_B", "")   # symbol of second interacting protein
        score  = entry.get("score", 0)              # combined confidence score (0–1 float)
        if gene_a and gene_b:                        # skip malformed entries
            interactions.append((gene_a, gene_b, score))

    return interactions


# ── Step 1: Collect all gene symbols from the dataset ─────────────────────────
# We only query STRING for genes already in the dataset (from KEGG and ChEMBL).
# This means STRING will only return interactions between genes we already know are relevant:
# It will not introduce new gene nodes that are not connected to any disease or pathway in the graph.

gene_symbols = [
    n["name"] for n in nodes
    if n["node_type"] == "Gene"    # only Gene-type nodes
]

print(f"Querying STRING for {len(gene_symbols)} gene symbols...")
print(f"Threshold: 0.9 (very high confidence, STRING score ≥ 900/1000)")
print(f"Database: STRING v12.0 | Species: Homo sapiens (9606)")
print()

# ── Step 2: Fetch interactions from STRING API ────────────────────────────────
# The full gene list is sent in one request. STRING handles large inputs by internally batching. 
# We add a short pause after the call as a courtesy.

interactions = get_string_interactions(gene_symbols, score_threshold=900)
time.sleep(1)   # brief pause after the API call

print(f"STRING returned {len(interactions)} interactions at threshold 0.9")
print()

# ── Step 3: Add valid interactions as INTERACTS_WITH edges ────────────────────
# We only add an edge if BOTH genes already exist in our dataset as nodes.
# This keeps the graph closed — STRING cannot introduce new nodes here.
# Each edge stores the combined confidence score in its notes field,
# which will appear in the hover tooltip during visualisation.

existing_ids = {n["node_id"] for n in nodes}   # set of all current node IDs for fast lookup

added   = 0   # counter for edges successfully added
skipped = 0   # counter for edges skipped because one gene was not in our dataset

for gene_a, gene_b, score in interactions:
    node_a = f"Gene:{gene_a}"   # construct node ID matching our naming convention
    node_b = f"Gene:{gene_b}"   # construct node ID matching our naming convention

    if node_a in existing_ids and node_b in existing_ids:
        # Both genes exist in the dataset — safe to add the edge
        add_edge(
            source_id  = node_a,
            relation   = "INTERACTS_WITH",
            target_id  = node_b,
            source_db  = "STRING",
            source_ref = f"https://string-db.org/network/{gene_a}%0d{gene_b}",
            notes      = f"STRING v12.0 combined score: {score:.3f} (threshold ≥ 0.9)"
        )
        added += 1
    else:
        skipped += 1   # one or both genes not in the dataset — skip this interaction

# ── Step 4: Summary ───────────────────────────────────────────────────────────
print(f"INTERACTS_WITH edges added:   {added}")
print(f"Interactions skipped (gene not in dataset): {skipped}")
print(f"Total edges now: {len(edges)}")
print()
print("Cell 2 complete. Ready for Cell 3: save the file.")

Querying STRING for 157 gene symbols...
Threshold: 0.9 (very high confidence, STRING score ≥ 900/1000)
Database: STRING v12.0 | Species: Homo sapiens (9606)

STRING returned 651 interactions at threshold 0.9

INTERACTS_WITH edges added:   639
Interactions skipped (gene not in dataset): 12
Total edges now: 1027

Cell 2 complete. Ready for Cell 3: save the file.


In [18]:
# CELL 3 — Export updated dataset to Excel
#
# PURPOSE:
#   Save the current nodes and edges lists to a new Excel file before
#   proceeding with any graph analysis. This preserves the cleaned dataset
#   (original KEGG/ChEMBL data + STRING edges re-fetched at threshold 0.9)
#   as a standalone file that can be reloaded at any point without having
#   to re-run the API calls.
#
#   We save to a NEW filename (auto6) rather than overwriting auto5, so the
#   previous version is preserved as a backup. This follows the versioning
#   convention of the original pipeline.
#
# INPUT:  nodes (list of dicts), edges (list of dicts) from Cells 1 and 2
# OUTPUT: T2DM_Melanoma_KG_Dataset_auto6.xlsx (Sheet1 = nodes, Sheet2 = edges)


# ── Step 1: Convert nodes and edges to DataFrames ─────────────────────────────
# pd.DataFrame() converts our list of dicts directly into a table.
# Each key in the dict becomes a column; each dict becomes one row.

nodes_df = pd.DataFrame(nodes)   # 290 rows, one per node
edges_df = pd.DataFrame(edges)   # 1,027 rows, one per edge (388 KEGG/ChEMBL + 639 STRING at 0.9)

# ── Step 2: Print a quick summary before saving ───────────────────────────────
# Confirming counts here catches any accidental data loss before writing to disk.

print("=== PRE-SAVE SUMMARY ===")
print(f"Nodes to save: {len(nodes_df)}")
print(f"Edges to save: {len(edges_df)}")
print()
print("Node type breakdown:")
print(nodes_df["node_type"].value_counts().to_string())
print()
print("Edge relation breakdown:")
print(edges_df["relation"].value_counts().to_string())
print()

# ── Step 3: Write to Excel ────────────────────────────────────────────────────
# pd.ExcelWriter with the "with" statement ensures the file is properly
# closed and saved even if an error occurs mid-write.
# engine="openpyxl" is required for .xlsx format.

output_filename = "T2DM_Melanoma_KG_Dataset_auto6.xlsx"

with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    nodes_df.to_excel(writer, sheet_name="Sheet1", index=False)   # nodes → Sheet1
    edges_df.to_excel(writer, sheet_name="Sheet2", index=False)   # edges → Sheet2

print(f"Saved: {output_filename}")
print(f"  Sheet1 (nodes): {len(nodes_df)} rows")
print(f"  Sheet2 (edges): {len(edges_df)} rows")
print()
print("Cell 3 complete. Dataset saved. Ready for Cell 4: build NetworkX graph.")

=== PRE-SAVE SUMMARY ===
Nodes to save: 290
Edges to save: 1027

Node type breakdown:
node_type
Gene          157
Drug          102
Pathway        20
Metabolite      7
Disease         4

Edge relation breakdown:
relation
INTERACTS_WITH      639
HAS_GENE            152
HAS_DRUG            125
TARGETS              80
RELATED_PATHWAY      20
HAS_METABOLITE        7
SAME_AS               2
HAS_KEGG_PATHWAY      2

Saved: T2DM_Melanoma_KG_Dataset_auto6.xlsx
  Sheet1 (nodes): 290 rows
  Sheet2 (edges): 1027 rows

Cell 3 complete. Dataset saved. Ready for Cell 4: build NetworkX graph.


In [19]:
# CELL 4 — Build the NetworkX graph
#
# PURPOSE:
#   Convert the nodes and edges lists into a NetworkX directed graph (DiGraph).
#   NetworkX is the Python library used for all graph analysis in this project
#   (betweenness centrality, shortest path calculations for drug proximity scoring). 
#   Building the graph here once means Cells 5, 6, and 7 can all
#   operate on the same G object without rebuilding it each time.
#
#   We build a directed graph (DiGraph) rather than an undirected graph because
#   biological relationships are directional — a pathway HAS_GENE a gene is
#   not the same as a gene HAS_GENE a pathway. However, for certain analyses
#   (betweenness centrality, shortest path) we will use an undirected view
#   of the graph, since protein interactions and proximity are not directional
#   concepts. This is handled in Cells 5 and 6.
#
# INPUT:  T2DM_Melanoma_KG_Dataset_auto6.xlsx (auto6 = STRING at 0.9)
# OUTPUT: G (NetworkX DiGraph), G_undirected (undirected view for analysis)


from collections import Counter

# ── Step 1: Load the saved auto6 dataset ─────────────────────────────────────
# We load from the saved Excel file rather than relying on the nodes/edges
# variables in memory. This ensures the graph is always built from the
# verified, saved dataset — protecting against cell execution order errors.

nodes_df = pd.read_excel("T2DM_Melanoma_KG_Dataset_auto6.xlsx", sheet_name="Sheet1")
edges_df = pd.read_excel("T2DM_Melanoma_KG_Dataset_auto6.xlsx", sheet_name="Sheet2")

nodes = nodes_df.to_dict(orient="records")
edges = edges_df.to_dict(orient="records")

print(f"Loaded from auto6: {len(nodes)} nodes, {len(edges)} edges")


# ── Step 2: Initialise a directed graph ───────────────────────────────────────
# DiGraph = directed graph, meaning each edge has a source and a target.
# This matches the biology: relationships like HAS_GENE and TARGETS are
# directional. INTERACTS_WITH is bidirectional in biology but stored
# directionally here — the undirected view in Step 4 handles this for analysis.

G = nx.DiGraph()


# ── Step 3: Add all nodes with their attributes ───────────────────────────────
# Each node is added with all its metadata stored as node attributes.
# These attributes are used in:
#   - Visualisation (colour, label, hover tooltip)
#   - Analysis (filtering by node_type)
#   - Results reporting (name, notes)

for row in nodes:
    G.add_node(
        row["node_id"],
        node_type  = row["node_type"],
        name       = row["name"],
        source_db  = row.get("source_db", ""),
        source_ref = row.get("source_ref", ""),
        notes      = str(row.get("notes", "")),
    )


# ── Step 4: Add all edges with their attributes ───────────────────────────────
# Each edge stores its relation type and provenance.
# Edges where the source or target node is missing are skipped defensively.

skipped_edges = 0

for row in edges:
    src = row["source_id"]
    tgt = row["target_id"]

    if src not in G or tgt not in G:
        skipped_edges += 1
        continue

    G.add_edge(
        src, tgt,
        relation   = row.get("relation", ""),
        source_db  = row.get("source_db", ""),
        source_ref = row.get("source_ref", ""),
        notes      = str(row.get("notes", "")),
    )

if skipped_edges > 0:
    print(f"  Warning: {skipped_edges} edges skipped — source or target node not found")


# ── Step 5: Build an undirected view for analysis ─────────────────────────────
# Betweenness centrality and drug proximity scoring both require an
# undirected graph because we want the shortest route between nodes
# regardless of edge direction.
# G.to_undirected() creates a copy with all edges made bidirectional,
# preserving all node and edge attributes.

G_undirected = G.to_undirected()


# ── Step 6: Print a full summary of the graph ─────────────────────────────────

print()
print("=== KNOWLEDGE GRAPH SUMMARY ===")
print(f"Nodes:              {G.number_of_nodes()}")
print(f"Edges (directed):   {G.number_of_edges()}")
print()

# Node type breakdown
print("Node types:")
type_counts = Counter(G.nodes[n]["node_type"] for n in G.nodes)
for ntype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ntype:<12} {count}")
print()

# Edge relation breakdown
print("Edge relations:")
rel_counts = Counter(G.edges[u, v]["relation"] for u, v in G.edges)
for rel, count in sorted(rel_counts.items(), key=lambda x: -x[1]):
    print(f"  {rel:<25} {count}")
print()

# Connectivity check — critical for shortest path analysis in Cell 5
# A fully connected graph means every node can reach every other node,
# which is required for drug proximity scoring to work for all drugs.
n_components = nx.number_connected_components(G_undirected)
print(f"Connected components (undirected): {n_components}")
if n_components == 1:
    print("  Graph is fully connected — all nodes reachable for proximity analysis.")
else:
    print("  Graph has multiple components — some nodes unreachable for proximity scoring.")
    component_sizes = sorted(
        [len(c) for c in nx.connected_components(G_undirected)],
        reverse=True
    )
    print(f"  Component sizes (largest first): {component_sizes[:10]}")

print()
print("Cell 4 complete. G and G_undirected ready for Cells 5, 6, and 7.")

Loaded from auto6: 290 nodes, 1027 edges

=== KNOWLEDGE GRAPH SUMMARY ===
Nodes:              290
Edges (directed):   1027

Node types:
  Gene         157
  Drug         102
  Pathway      20
  Metabolite   7
  Disease      4

Edge relations:
  INTERACTS_WITH            639
  HAS_GENE                  152
  HAS_DRUG                  125
  TARGETS                   80
  RELATED_PATHWAY           20
  HAS_METABOLITE            7
  SAME_AS                   2
  HAS_KEGG_PATHWAY          2

Connected components (undirected): 1
  Graph is fully connected — all nodes reachable for proximity analysis.

Cell 4 complete. G and G_undirected ready for Cells 5, 6, and 7.


In [20]:
# CELL 5 — Full interactive knowledge graph visualisation (PyVis HTML)
#
# PURPOSE:
#   Generate an interactive HTML visualisation of the complete knowledge graph.
#   This version colours edges by relationship type and displays the relation
#   label directly on the edge when hovered or clicked.
#
#   Edge colour scheme:
#     HAS_GENE         → orange  (#ffa94d) — pathway/disease to gene
#     HAS_DRUG         → blue    (#63b3ed) — pathway/disease to drug
#     TARGETS          → red     (#ff4757) — drug to gene target
#     INTERACTS_WITH   → white   (rgba, semi-transparent) — STRING PPI
#     RELATED_PATHWAY  → yellow  (#f6c90e) — disease to related pathway
#     HAS_METABOLITE   → purple  (#da77f2) — pathway to metabolite
#     SAME_AS          → grey    (#aaaaaa) — seed to KEGG disease node
#     HAS_KEGG_PATHWAY → teal    (#63e6be) — disease to main pathway
#
#   Node colour scheme:
#     Disease         → red       (#ff4757)
#     Pathway         → orange    (#ffa94d)
#     Drug            → blue      (#63b3ed)
#     Gene            → green     (#68d391)
#     Metabolite      → purple    (#da77f2)
#     Shared gene     → yellow    (#f6c90e) — in BOTH pathway diagrams
#     Cross-disease   → white     (#ffffff) — bridges disease entry to
#                                             opposite disease pathway


from pyvis.network import Network   # interactive HTML graph visualisation
from collections import Counter     # for counting node types in summary


# ── Step 1: Define colour maps ────────────────────────────────────────────────
# Colours are chosen to be visually distinct from each other and consistent
# with the node colours where applicable (e.g. HAS_GENE edges are orange
# like Pathway nodes, TARGETS edges are red like Disease nodes).
# This makes it easier to follow the biology when exploring the graph.

NODE_COLOR_MAP = {
    "Disease":    "#e63946",   # strong red — anchor disease nodes
    "Pathway":    "#e07b00",   # dark orange — KEGG biological pathways
    "Drug":       "#1d6fa4",   # dark blue — T2DM pharmacological agents
    "Gene":       "#2d8a4e",   # dark green — gene/protein nodes
    "Metabolite": "#7b2fa8",   # dark purple — KEGG pathway metabolites
}


SHARED_COLOR        = "#c9a800"   # dark yellow — genes shared across both pathway diagrams, visible on white
CROSS_DISEASE_COLOR = "#1a1a6e"   # dark navy — genes bridging disease entry to opposite pathway
DEFAULT_NODE_COLOR  = "#666666"   # mid grey - fallback for unrecognised node types

# Edge colours are keyed by the relation type string stored in each edge's
# "relation" attribute. Each relation type gets a distinct colour so the
# type of biological relationship is immediately visible in the graph.

EDGE_COLOR_MAP = {
    "HAS_GENE":         "#e07b00",                # dark orange — pathway or disease HAS_GENE a gene
    "HAS_DRUG":         "#1d6fa4",                # dark blue — pathway or disease HAS_DRUG a drug
    "TARGETS":          "#e63946",                # dark red — drug TARGETS a gene/protein
    "INTERACTS_WITH":   "rgba(150,150,150,0.4)",  # medium grey, more opaque — STRING PPI (very transparent
                                                   #               to reduce visual noise from
                                                   #               639 PPI edges)
    "RELATED_PATHWAY":  "#c9a800",                # dark yellow — disease RELATED_PATHWAY a pathway
    "HAS_METABOLITE":   "#7b2fa8",                # dark purple — pathway HAS_METABOLITE a compound
    "SAME_AS":          "#888888",                # medium grey — seed node SAME_AS KEGG disease node
    "HAS_KEGG_PATHWAY": "#00897b",                # teal — disease HAS_KEGG_PATHWAY its main pathway
}
DEFAULT_EDGE_COLOR = "#888888"   # grey fallback for any unrecognised relation type

# ── Step 2: Identify special gene categories ──────────────────────────────────
# We define two scientifically distinct categories of biologically important genes:
#
# (1) SHARED PATHWAY GENES (yellow):
#     Genes that appear in BOTH the T2DM pathway diagram (hsa04930) AND the
#     melanoma pathway diagram (hsa05218). These suggest shared signalling
#     machinery between the two diseases at the pathway level.
#     Examples: PIK3CA, PIK3CB, MAPK1, MAPK3
#
# (2) CROSS-DISEASE GENES (white):
#     Genes that appear in one disease's DISEASE ENTRY (H00409 or H00038)
#     AND the other disease's pathway. This is a stronger connection than
#     shared pathway genes because disease entries capture genes with direct
#     genetic evidence (GWAS, linkage studies), not just pathway membership.
#     Example: AKT2 — listed in T2DM disease entry H00409 AND melanoma pathway hsa05218

# Genes listed directly in the T2DM DISEASE entry (H00409)
# These come from genetic/GWAS evidence, not just the pathway diagram
t2dm_disease_genes = {
    v for u, v, d in G.edges(data=True)
    if u == "KEGG:Disease:H00409" and d["relation"] == "HAS_GENE"
}

# Genes listed directly in the Melanoma DISEASE entry (H00038)
mel_disease_genes = {
    v for u, v, d in G.edges(data=True)
    if u == "KEGG:Disease:H00038" and d["relation"] == "HAS_GENE"
}

# Genes in the T2DM pathway diagram (hsa04930)
t2dm_pathway_genes = {
    v for u, v, d in G.edges(data=True)
    if u == "KEGG:Pathway:hsa04930" and d["relation"] == "HAS_GENE"
}

# Genes in the Melanoma pathway diagram (hsa05218)
mel_pathway_genes = {
    v for u, v, d in G.edges(data=True)
    if u == "KEGG:Pathway:hsa05218" and d["relation"] == "HAS_GENE"
}

# Set intersection — genes present in both pathway diagrams simultaneously
shared_pathway_genes = t2dm_pathway_genes & mel_pathway_genes

# Union of all four cross-disease combinations, minus any already classified
# as shared pathway genes (to avoid double-counting):
#   T2DM disease entry ∩ Melanoma pathway  → catches AKT2
#   Melanoma disease entry ∩ T2DM pathway  → catches any equivalent reverse case
#   T2DM disease entry ∩ Melanoma disease entry → genes in both disease entries
cross_disease_genes = (
    (t2dm_disease_genes & mel_pathway_genes)  |   # T2DM disease → Melanoma pathway
    (mel_disease_genes  & t2dm_pathway_genes) |   # Melanoma disease → T2DM pathway
    (t2dm_disease_genes & mel_disease_genes)      # both disease entries directly
) - shared_pathway_genes   # exclude genes already in shared pathway category

# Print identified genes so the user can verify the classification is correct
print("=== SPECIAL GENE CATEGORIES ===")
print(f"Shared pathway genes (yellow): {len(shared_pathway_genes)}")
for g in sorted(shared_pathway_genes):
    print(f"  {G.nodes[g]['name']}")   # print human-readable gene name
print()
print(f"Cross-disease genes (white):   {len(cross_disease_genes)}")
for g in sorted(cross_disease_genes):
    print(f"  {G.nodes[g]['name']}")
print()


# ── Step 3: Calculate degree-based node sizing ────────────────────────────────
# Node size in the visualisation reflects degree (number of connected edges).
# Degree is a simple and intuitive measure: highly connected nodes (hubs like
# pathway nodes and hub genes) appear larger, making them immediately visible.
#
# Sizes are scaled linearly between MIN_SIZE and MAX_SIZE.
# Special nodes (shared genes, cross-disease genes) get a guaranteed minimum
# size of MIN_SPECIAL so they are never too small to see in the dense gene cluster.

degree      = dict(G_undirected.degree())   # {node_id: degree} for all nodes
MIN_SIZE    = 10     # smallest any node can appear (low-degree nodes like metabolites)
MAX_SIZE    = 60     # largest any node can appear (highest-degree pathway/disease hubs)
MIN_SPECIAL = 30     # minimum size for highlighted genes — prevents them disappearing
max_degree  = max(degree.values()) if degree else 1   # normalisation denominator

def scale_size(node_id, is_special=False):
    """
    Scale a node's degree to a visual size between MIN_SIZE and MAX_SIZE.
    If is_special=True, apply a minimum size floor of MIN_SPECIAL so that
    biologically important nodes (shared genes, cross-disease genes) remain
    visible even when their degree is relatively low.
    """
    d      = degree.get(node_id, 1)
    scaled = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * (d / max_degree)
    return max(scaled, MIN_SPECIAL) if is_special else scaled


# ── Step 4: Initialise PyVis network ──────────────────────────────────────────
# We do NOT use net.set_options() — passing a custom JSON options block
# replaces ALL default PyVis settings including the default edge tooltip
# behaviour. The original notebook used no set_options() and tooltips worked
# correctly. We replicate that approach here.

net = Network(
    height     = "900px",
    width      = "100%",
    bgcolor    = "#ffffff",    # white background
    font_color = "#222222",    # dark text for readability on white
    directed   = True,
)

# Physics layout — native barnes_hut method only, no set_options()
net.barnes_hut(
    gravity         = -12000,
    central_gravity = 0.2,
    spring_length   = 220,
    spring_strength = 0.04,
    damping         = 0.09,
)


# ── Step 5: Add all nodes to the network ──────────────────────────────────────
# Each node is assigned:
#   label  — short name displayed on the node in the graph
#   color  — type-based colour, with special overrides for shared/cross-disease genes
#   size   — scaled by degree, with a minimum floor for highlighted genes
#   title  — HTML tooltip shown when hovering over the node (all metadata)

for node_id, attr in G.nodes(data=True):
    ntype  = attr.get("node_type", "Gene")
    name   = attr.get("name", node_id)
    notes  = str(attr.get("notes", "") or "")      # cast NaN → empty string
    source = str(attr.get("source_db", "") or "")  # cast NaN → empty string
    ref    = str(attr.get("source_ref", "") or "")  # cast NaN → empty string
    deg    = degree.get(node_id, 0)

    # Determine node colour — priority order:
    # 1. Cross-disease gene (white) — highest scientific priority
    # 2. Shared pathway gene (yellow) — second priority
    # 3. Type-based colour — default
    is_special = False
    if node_id in cross_disease_genes:
        color      = CROSS_DISEASE_COLOR
        is_special = True
        category   = "Cross-disease gene (bridges T2DM disease ↔ Melanoma pathway)"
    elif node_id in shared_pathway_genes or "Shared" in notes:
        color      = SHARED_COLOR
        is_special = True
        category   = "Shared pathway gene (hsa04930 ∩ hsa05218)"
    else:
        color    = NODE_COLOR_MAP.get(ntype, DEFAULT_NODE_COLOR)
        category = ntype   # use node type as the category label for tooltip

    # Build the HTML tooltip — shown when the user hovers over a node
    # Each field is on a separate line for readability in the popup
    tooltip = (
        f"<b>{name}</b><br>"
        f"Category: {category}<br>"
        f"Node type: {ntype}<br>"
        f"Degree: {deg}<br>"
        f"Source: {source}<br>"
        f"Notes: {notes}<br>"
        f"Ref: <a href='{ref}' target='_blank'>"
        f"{ref[:60]}{'...' if len(ref) > 60 else ''}</a>"
    )

    net.add_node(
        node_id,
        label = name,                        # short label on the node
        color = color,                       # type/category colour
        size  = scale_size(node_id, is_special),  # degree-based size
        title = tooltip,                     # hover tooltip with full metadata
    )


# ── Step 6: Add all edges coloured and labelled by relation type ──────────────
# PyVis uses "title" to display a tooltip popup when hovering over an edge.
# This is the correct parameter for hover-based relationship display — it
# matches the behaviour in the original notebook (title=attr.get("relation",""))
# and is what produces the popup shown in the reference screenshot.
#
# "label" draws permanent text on the edge line which becomes unreadable in a
# dense graph — we do not use it here.
#
# INTERACTS_WITH (STRING PPI) edges are drawn faintly since there are 639 of
# them and they would dominate the visual if drawn at full opacity.

for u, v, attr in G.edges(data=True):
    relation = attr.get("relation", "")    # e.g. "HAS_GENE", "TARGETS", "INTERACTS_WITH"
    source   = attr.get("source_db", "")  # e.g. "KEGG", "ChEMBL", "STRING"
    notes    = str(attr.get("notes", "") or "")

    # Look up the colour for this relation type
    edge_color = EDGE_COLOR_MAP.get(relation, DEFAULT_EDGE_COLOR)

    # Title parameter creates the hover tooltip popup — this is the correct
    # PyVis mechanism for showing edge information on hover/click,
    # matching the behaviour seen in the reference screenshot
    edge_tooltip = (
        f"Relation: {relation}\n"
        f"Source: {source}\n"
        f"Notes: {notes}"
    )

    if relation == "INTERACTS_WITH":
        # PPI edges — medium grey, thin but visible on white background
        net.add_edge(u, v,
                     title  = edge_tooltip,   # hover tooltip shows relation info
                     color  = edge_color,
                     width  = 0.8)       
    else:
        # KEGG and ChEMBL edges — coloured by type
        net.add_edge(u, v,
                     title  = edge_tooltip,   # hover tooltip shows full relation info
                     color  = edge_color,
                     width  = 2.0)       

        
# ── Step 7: Configure interaction and display options ─────────────────────────
# These options are passed directly to the vis.js library that PyVis uses
# under the hood. They control how the graph responds to user interaction.

net.set_options("""
var options = {
  "nodes": {
    "font": {"size": 11, "face": "Arial"},
    "borderWidth": 1.5,
    "borderWidthSelected": 3
  },
  "edges": {
    "smooth": {"type": "continuous"},
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.5}},
    "hoverWidth": 3,
    "selectionWidth": 3
  },
  "interaction": {
    "hover": true,
    "hoverConnectedEdges": true,
    "tooltipDelay": 100,
    "navigationButtons": true,
    "keyboard": true,
    "selectConnectedEdges": true
  },
  "physics": {
    "enabled": true,
    "stabilization": {"iterations": 200}
  }
}
""")


# ── Step 8: Save the HTML file ────────────────────────────────────────────────
# The output is a single self-contained HTML file that can be opened in
# any modern browser without any additional software or internet connection.
# All the graph data and the vis.js library are embedded in the file.

output_file = "kg_full_v1.html"
net.write_html(output_file)

print(f"Saved: {output_file}")
print()
print("=== EDGE COLOUR LEGEND ===")
for rel, color in EDGE_COLOR_MAP.items():
    print(f"  {color:<40} {rel}")   # print relation and its colour for reference
print()
print("Cell 5 complete. Proceed to Cell 6: betweenness centrality analysis.")


# ── Step 9: Post-process HTML to inject reliable edge tooltip handler ─────────
# PyVis's built-in edge title/tooltip system is unreliable across versions.
# The guaranteed solution is to inject a vis.js JavaScript event listener
# directly into the generated HTML after PyVis writes it.
# This uses the vis.js network.on("hoverEdge") API which is always reliable
# regardless of PyVis version or options configuration.
#
# The injected script:
#   1. Creates a floating tooltip div styled to match the dark graph theme
#   2. Listens for the hoverEdge event and reads the edge's title attribute
#   3. Positions the tooltip near the cursor
#   4. Hides the tooltip when the cursor leaves the edge (blurEdge event)

# Read the HTML file PyVis just wrote
with open(output_file, "r", encoding="utf-8") as f:
    html = f.read()

# JavaScript to inject — adds a floating edge tooltip using vis.js events
edge_tooltip_js = """
<style>
  #edge-tooltip {
    position: fixed;
    background: #1a1a2e;
    color: white;
    padding: 8px 12px;
    border-radius: 6px;
    font-size: 12px;
    font-family: Arial, sans-serif;
    border: 1px solid #ffa94d;
    pointer-events: none;
    z-index: 9999;
    display: none;
    max-width: 300px;
    line-height: 1.5;
  }
</style>
<div id="edge-tooltip"></div>
<script>
  // Wait for the vis.js network object to be available
  // PyVis names it "network" in the generated HTML
  window.addEventListener("load", function() {
    var tooltip = document.getElementById("edge-tooltip");

    // Fired when the cursor moves onto an edge
    network.on("hoverEdge", function(params) {
      var edgeId  = params.edge;
      var edgeObj = network.body.data.edges.get(edgeId);

      if (edgeObj && edgeObj.title) {
        // Replace newlines with HTML line breaks for display
        tooltip.innerHTML = edgeObj.title.replace(/\\n/g, "<br>");
        tooltip.style.display = "block";
      }

      // Position tooltip near the cursor using the mouse event coordinates
      var mouseEvent = params.event;
      var x = mouseEvent.pageX || (mouseEvent.center ? mouseEvent.center.x : 0);
      var y = mouseEvent.pageY || (mouseEvent.center ? mouseEvent.center.y : 0);
      tooltip.style.left = (x + 15) + "px";
      tooltip.style.top  = (y - 10) + "px";
    });

    // Fired when the cursor leaves an edge — hide the tooltip
    network.on("blurEdge", function() {
      tooltip.style.display = "none";
    });

    // Also hide tooltip when the canvas is clicked (cleanup)
    network.on("click", function() {
      tooltip.style.display = "none";
    });
  });
</script>
"""

# Inject the tooltip script just before the closing </body> tag
# This ensures it runs after PyVis has fully initialised the network object
if "</body>" in html:
    html = html.replace("</body>", edge_tooltip_js + "\n</body>")
else:
    # Fallback — append at end of file if </body> not found
    html = html + edge_tooltip_js

# Write the modified HTML back to the same file
with open(output_file, "w", encoding="utf-8") as f:
    f.write(html)


=== SPECIAL GENE CATEGORIES ===
Shared pathway genes (yellow): 9
  MAPK1
  MAPK3
  P3R3URF-PIK3R3
  PIK3CA
  PIK3CB
  PIK3CD
  PIK3R1
  PIK3R2
  PIK3R3

Cross-disease genes (white):   1
  AKT2

Saved: kg_full_v1.html

=== EDGE COLOUR LEGEND ===
  #e07b00                                  HAS_GENE
  #1d6fa4                                  HAS_DRUG
  #e63946                                  TARGETS
  rgba(150,150,150,0.4)                    INTERACTS_WITH
  #c9a800                                  RELATED_PATHWAY
  #7b2fa8                                  HAS_METABOLITE
  #888888                                  SAME_AS
  #00897b                                  HAS_KEGG_PATHWAY

Cell 5 complete. Proceed to Cell 6: betweenness centrality analysis.


In [21]:
# CELL 6 — Betweenness centrality analysis
#
# PURPOSE:
#   Calculate betweenness centrality for every node in the knowledge graph.
#
#   Betweenness centrality measures how frequently a node lies on the shortest
#   path between all other pairs of nodes in the network. A node with high
#   betweenness is a structural "bridge" — information, signals, or molecular
#   interactions between distant parts of the network must pass through it.
#   Removing such a node would significantly disrupt connectivity.
#
#   In a biomedical knowledge graph, high-betweenness gene nodes represent
#   mechanistic bottlenecks: proteins that mediate signal flow between disease
#   modules, pathway clusters, and drug targets. These are the most
#   biologically significant nodes for understanding how T2DM and melanoma
#   are molecularly connected.
#
# FORMULA:
#   BC(v) = Σ (σ_st(v) / σ_st) for all pairs s ≠ v ≠ t
#   where:
#     σ_st     = total number of shortest paths between nodes s and t
#     σ_st(v)  = number of those shortest paths that pass through node v
#   Scores are normalised to [0,1] by NetworkX by default, making them
#   comparable across graphs of different sizes.
#
# GRAPH USED:
#   G_undirected — betweenness centrality is computed on the undirected graph
#   because we want to capture structural importance regardless of the edge
#   direction. A gene is equally important as a bridge, whether the edges point
#   toward or away from it.
#
# INPUT:  G, G_undirected (from Cell 3)
# OUTPUT: centrality_df  — DataFrame of all 290 nodes ranked by score
#         Results printed to screen and stored as node attributes on G


import pandas as pd
from collections import Counter

print("Calculating betweenness centrality for all 290 nodes...")
print()

# ── Step 1: Calculate betweenness centrality ──────────────────────────────────
# nx.betweenness_centrality() computes normalised scores for every node
# simultaneously using Brandes' algorithm.
#   normalized=True  — scales scores to [0,1] for cross-graph comparability
#   weight=None      — all edges treated as equal weight (unweighted graph)
#                      We do not weight by STRING confidence score here because
#                      KEGG and ChEMBL edges have no equivalent numeric weight,
#                      so weighting would create an inconsistent comparison.

centrality = nx.betweenness_centrality(
    G_undirected,
    normalized = True,    # normalise scores to [0,1]
    weight     = None,    # unweighted — all edges treated equally
)
# centrality is a dict: {node_id: score}
# e.g. {"Gene:MAPK1": 0.0412, "KEGG:Pathway:hsa05218": 0.0891, ...}


# ── Step 2: Store scores as node attributes ───────────────────────────────────
# Storing the score directly on each node makes it accessible during
# visualisation (Cell 7 poster subgraph uses centrality for node sizing)
# and during drug proximity scoring (Cell 6).

for node_id, score in centrality.items():
    if node_id in G.nodes:
        G.nodes[node_id]["betweenness"] = round(score, 6)
    if node_id in G_undirected.nodes:
        G_undirected.nodes[node_id]["betweenness"] = round(score, 6)


# ── Step 3: Build a results DataFrame ─────────────────────────────────────────
# Combine centrality scores with node metadata into a sortable table.
# This is the format used for reporting results in the dissertation.

records = []
for node_id, score in centrality.items():
    attr = G.nodes[node_id]
    records.append({
        "node_id":     node_id,
        "name":        attr.get("name", node_id),
        "node_type":   attr.get("node_type", ""),
        "betweenness": round(score, 6),
        "degree":      G_undirected.degree(node_id),   # include degree for comparison
        "notes":       attr.get("notes", ""),
    })

# Sort by betweenness score descending
centrality_df = pd.DataFrame(records).sort_values(
    "betweenness", ascending=False
).reset_index(drop=True)


# ── Step 4: Print results ─────────────────────────────────────────────────────

print("=== TOP 20 NODES BY BETWEENNESS CENTRALITY (all types) ===")
print(centrality_df[["name", "node_type", "betweenness", "degree"]].head(20).to_string(index=False))
print()

# Gene nodes specifically — these are the mechanistic bottlenecks
# that are most relevant to the dissertation findings
print("=== TOP 15 GENE NODES BY BETWEENNESS CENTRALITY ===")
gene_df = centrality_df[centrality_df["node_type"] == "Gene"].head(15)
print(gene_df[["name", "betweenness", "degree", "notes"]].to_string(index=False))
print()

# Highlight shared and cross-disease genes specifically
# These are the most biologically significant nodes in the context
# of the T2DM-melanoma interface
print("=== SHARED AND CROSS-DISEASE GENES — CENTRALITY RANKING ===")
special_df = centrality_df[
    centrality_df["notes"].str.contains("Shared|Cross", case=False, na=False) |
    centrality_df["node_id"].isin(cross_disease_genes)
].copy()

if len(special_df) > 0:
    print(special_df[["name", "node_type", "betweenness", "degree", "notes"]].to_string(index=False))
else:
    # Fallback — look up the shared genes manually by node_id
    shared_ids  = list(shared_pathway_genes) + list(cross_disease_genes)
    special_df  = centrality_df[centrality_df["node_id"].isin(shared_ids)]
    print(special_df[["name", "node_type", "betweenness", "degree"]].to_string(index=False))
print()

# Summary statistics across all nodes
print("=== CENTRALITY SUMMARY STATISTICS ===")
print(f"Highest score:      {centrality_df['betweenness'].max():.6f}  "
      f"({centrality_df.iloc[0]['name']})")
print(f"Mean score:         {centrality_df['betweenness'].mean():.6f}")
print(f"Median score:       {centrality_df['betweenness'].median():.6f}")
print(f"Nodes with BC > 0:  {(centrality_df['betweenness'] > 0).sum()} "
      f"of {len(centrality_df)}")
print()
print("Cell 6 complete. Proceed to Cell 7: drug proximity scoring.")

Calculating betweenness centrality for all 290 nodes...

=== TOP 20 NODES BY BETWEENNESS CENTRALITY (all types) ===
                   name node_type  betweenness  degree
                   T2DM   Disease     0.446230      97
Type II diabetes mellit   Pathway     0.271792      93
               Melanoma   Pathway     0.217133      87
               Melanoma   Disease     0.128183      31
                   AKT2      Gene     0.077959      21
                   TP53      Gene     0.074282      34
                    INS      Gene     0.072291      36
                   IRS1      Gene     0.066258      29
                   EGFR      Gene     0.028722      42
                    TNF      Gene     0.027196       6
             Cell cycle   Pathway     0.025505       2
  p53 signaling pathway   Pathway     0.025505       2
                    GCK      Gene     0.024427      14
                  PPARG      Gene     0.023432       6
                    CD4      Gene     0.021701       5
    

In [22]:
# CELL 7 — Drug proximity scoring
#
# PURPOSE:
#   Rank all T2DM drugs in the knowledge graph by their mechanistic proximity
#   to melanoma biology. Proximity is defined as the mean shortest path length
#   between a drug's target genes and the set of melanoma pathway genes in the
#   undirected protein interaction network.
#
#   This follows the network-based drug proximity framework established by
#   Cheng et al. (2018, Nature Communications), which demonstrated that drugs
#   whose targets are network-proximal to disease genes are significantly more
#   likely to have therapeutic effects on that disease than drugs whose targets
#   are distant.
#
#   A lower proximity score = shorter average path = mechanistically closer
#   to melanoma biology = higher biological relevance.
#
# METHOD:
#   For each drug node:
#     1. Identify its target gene nodes via TARGETS edges
#     2. Identify all melanoma pathway gene nodes via HAS_GENE edges
#        from KEGG:Pathway:hsa05218
#     3. For each (drug target, melanoma gene) pair, compute the shortest
#        path length in G_undirected
#     4. Average all shortest path lengths → proximity score for that drug
#     5. Drugs with no targets or no reachable melanoma genes are flagged
#        separately rather than excluded silently
#
#   Lower score = closer to melanoma = higher mechanistic proximity
#   Score of 1.0 = drug directly targets a melanoma pathway gene
#
# INPUT:  G, G_undirected (from Cell 3)
# OUTPUT: proximity_df — DataFrame of drugs ranked by proximity score


import pandas as pd
import numpy as np    # for mean calculation and handling missing values

print("Calculating drug proximity scores to melanoma biology...")
print("This computes shortest paths between all drug targets and melanoma genes.")
print()

# ── Step 1: Define the melanoma gene set ──────────────────────────────────────
# The melanoma gene set is all genes connected to the melanoma pathway node
# via HAS_GENE edges. These are the molecular targets of interest — drugs
# whose targets are close to these genes in the network are mechanistically
# proximal to melanoma biology.

melanoma_genes = {
    v for u, v, d in G.edges(data=True)
    if u == "KEGG:Pathway:hsa05218" and d["relation"] == "HAS_GENE"
}

# Also include melanoma disease entry genes for completeness
melanoma_disease_genes = {
    v for u, v, d in G.edges(data=True)
    if u == "KEGG:Disease:H00038" and d["relation"] == "HAS_GENE"
}

# Combined melanoma gene set — pathway genes + disease entry genes
melanoma_gene_set = melanoma_genes | melanoma_disease_genes

print(f"Melanoma gene set: {len(melanoma_gene_set)} genes")
print(f"  From pathway (hsa05218): {len(melanoma_genes)}")
print(f"  From disease entry (H00038): {len(melanoma_disease_genes)}")
print()


# ── Step 2: Build drug-target lookup ──────────────────────────────────────────
# For each drug node, find all its target genes via TARGETS edges.
# Drug nodes have IDs like "KEGG:Drug:D00944" (Metformin).
# Some drugs have no targets in the dataset (KEGG or ChEMBL returned nothing)
# — these will be flagged in the results rather than silently dropped.

drug_targets = {}   # {drug_node_id: [list of target gene node_ids]}

for u, v, d in G.edges(data=True):
    if d["relation"] == "TARGETS":
        # u = drug node, v = gene target node
        if u not in drug_targets:
            drug_targets[u] = []
        drug_targets[u].append(v)

# Get all drug nodes in the graph
all_drug_nodes = [n for n in G.nodes if G.nodes[n]["node_type"] == "Drug"]

print(f"Total drug nodes in graph: {len(all_drug_nodes)}")
print(f"Drugs with at least one target: {len(drug_targets)}")
print(f"Drugs with no targets: {len(all_drug_nodes) - len(drug_targets)}")
print()


# ── Step 3: Compute proximity scores ──────────────────────────────────────────
# For each drug, compute the mean shortest path from its targets to all
# melanoma genes. We use the undirected graph so path direction does not
# affect the calculation — we want the shortest route regardless of whether
# edges point toward or away from the target.
#
# nx.shortest_path_length(G, source, target) returns the integer length of
# the shortest path between two nodes. If no path exists (disconnected
# components), it raises NetworkXNoPath — we catch this and assign a large
# penalty value (999) so the drug sorts to the bottom of the ranking.

proximity_records = []   # will become the results DataFrame

for drug_node_id in all_drug_nodes:
    drug_name    = G.nodes[drug_node_id].get("name", drug_node_id)
    targets      = drug_targets.get(drug_node_id, [])   # empty list if no targets

    if not targets:
        # Drug has no targets in the dataset — cannot compute proximity
        # Record it with a flag rather than dropping it entirely
        proximity_records.append({
            "drug_node_id":   drug_node_id,
            "drug_name":      drug_name,
            "n_targets":      0,
            "mean_proximity": None,   # None = not computable
            "min_proximity":  None,
            "status":         "no_targets",
        })
        continue

    # Collect all pairwise shortest path lengths between drug targets
    # and melanoma genes
    path_lengths = []

    for target in targets:
        if target not in G_undirected:
            continue   # target node not in undirected graph — skip

        for mel_gene in melanoma_gene_set:
            if mel_gene not in G_undirected:
                continue   # melanoma gene not in undirected graph — skip

            if target == mel_gene:
                # Drug directly targets a melanoma gene — proximity = 1
                # (one step: drug → target which IS a melanoma gene)
                path_lengths.append(1)
                continue

            try:
                # Compute shortest path length between target and melanoma gene
                length = nx.shortest_path_length(G_undirected, target, mel_gene)
                path_lengths.append(length + 1)   # +1 to account for the drug→target edge
            except nx.NetworkXNoPath:
                # No path exists between this target and this melanoma gene
                # Assign large penalty so this pair does not inflate the score
                path_lengths.append(999)

    if path_lengths:
        mean_prox = round(float(np.mean(path_lengths)), 4)
        min_prox  = round(float(np.min(path_lengths)), 4)
        status    = "computed"
    else:
        mean_prox = None
        min_prox  = None
        status    = "no_paths"

    proximity_records.append({
        "drug_node_id":   drug_node_id,
        "drug_name":      drug_name,
        "n_targets":      len(targets),
        "mean_proximity": mean_prox,
        "min_proximity":  min_prox,
        "status":         status,
    })

# Build DataFrame — sort by mean proximity ascending (lower = closer)
proximity_df = pd.DataFrame(proximity_records)

# Separate drugs with computed scores from those without targets
scored_df  = proximity_df[proximity_df["status"] == "computed"].sort_values(
    "mean_proximity", ascending=True
).reset_index(drop=True)

no_target_df = proximity_df[proximity_df["status"] != "computed"]


# ── Step 4: Classify drugs by class ───────────────────────────────────────────
# Add a drug class column to the scored DataFrame so we can group results
# by therapeutic class (Metformin, GLP-1 RA, SGLT2i, TZD, etc.)
# This is done by keyword matching on the drug name.

def classify_drug(name):
    """Assign a pharmacological class label based on drug name keywords."""
    name_lower = name.lower()
    if any(x in name_lower for x in ["metformin"]):
        return "Biguanide (Metformin)"
    elif any(x in name_lower for x in ["semaglutide", "liraglutide", "exenatide",
                                        "dulaglutide", "albiglutide", "tirzepatide"]):
        return "GLP-1 Receptor Agonist"
    elif any(x in name_lower for x in ["dapagliflozin", "canagliflozin",
                                        "empagliflozin", "ertugliflozin",
                                        "bexagliflozin"]):
        return "SGLT2 Inhibitor"
    elif any(x in name_lower for x in ["pioglitazone", "rosiglitazone"]):
        return "Thiazolidinedione (TZD)"
    elif any(x in name_lower for x in ["sitagliptin", "saxagliptin", "linagliptin",
                                        "alogliptin", "teneligliptin",
                                        "trelagliptin", "omarigliptin",
                                        "anagliptin"]):
        return "DPP-4 Inhibitor"
    elif any(x in name_lower for x in ["glipizide", "glimepiride", "glyburide",
                                        "glibenclamide", "chlorpropamide",
                                        "tolbutamide", "tolazamide",
                                        "gliclazide", "gliquidone",
                                        "glisoxepide"]):
        return "Sulfonylurea"
    elif any(x in name_lower for x in ["insulin"]):
        return "Insulin"
    elif any(x in name_lower for x in ["repaglinide", "nateglinide",
                                        "mitiglinide"]):
        return "Meglitinide"
    elif any(x in name_lower for x in ["acarbose", "miglitol", "voglibose"]):
        return "Alpha-glucosidase inhibitor"
    elif any(x in name_lower for x in ["pramlintide"]):
        return "Amylin analogue"
    elif any(x in name_lower for x in ["colesevelam"]):
        return "Bile acid sequestrant"
    elif any(x in name_lower for x in ["ipilimumab", "nivolumab", "pembrolizumab",
                                        "atezolizumab", "talimogene",
                                        "interferon", "interleukin"]):
        return "Immunotherapy / Melanoma drug"
    elif any(x in name_lower for x in ["vemurafenib", "dabrafenib", "encorafenib",
                                        "trametinib", "cobimetinib",
                                        "binimetinib", "selumetinib"]):
        return "BRAF/MEK Inhibitor (Melanoma)"
    else:
        return "Other / Combination"

scored_df["drug_class"] = scored_df["drug_name"].apply(classify_drug)


# ── Step 5: Print results ─────────────────────────────────────────────────────

print("=== TOP 20 DRUGS BY PROXIMITY TO MELANOMA BIOLOGY ===")
print("(Lower score = mechanistically closer to melanoma)")
print()
print(scored_df[["drug_name", "drug_class", "n_targets",
                  "mean_proximity", "min_proximity"]].head(20).to_string(index=False))
print()

# Summary by drug class — mean proximity per class
print("=== MEAN PROXIMITY SCORE BY DRUG CLASS ===")
class_summary = (
    scored_df.groupby("drug_class")["mean_proximity"]
    .agg(["mean", "min", "count"])
    .rename(columns={"mean": "mean_proximity", "min": "best_proximity", "count": "n_drugs"})
    .sort_values("mean_proximity")
    .reset_index()
)
print(class_summary.to_string(index=False))
print()

print(f"Drugs with computed proximity scores: {len(scored_df)}")
print(f"Drugs with no targets (unscored):     {len(no_target_df)}")
print()
print("Cell 7 complete. Proceed to Cell 8: poster subgraph visualisation.")

Calculating drug proximity scores to melanoma biology...
This computes shortest paths between all drug targets and melanoma genes.

Melanoma gene set: 78 genes
  From pathway (hsa05218): 73
  From disease entry (H00038): 12

Total drug nodes in graph: 102
Drugs with at least one target: 63
Drugs with no targets: 39

=== TOP 20 DRUGS BY PROXIMITY TO MELANOMA BIOLOGY ===
(Lower score = mechanistically closer to melanoma)

                                                                         drug_name             drug_class  n_targets  mean_proximity  min_proximity
                                                           Insulin human (USP/INN)                Insulin          1          3.3462            2.0
                                                          Insulin lispro (USP/INN)                Insulin          1          3.3462            2.0
                                                          Insulin aspart (USP/INN)                Insulin          1          3.3462

In [23]:
# CELL 8 — Poster subgraph visualisation (PyVis HTML)
#
# PURPOSE:
#   Generate a focused, publication-quality interactive subgraph for the
#   poster. Unlike the full graph (Cell 5), which shows all 290 nodes, this
#   subgraph highlights only the most scientifically relevant nodes:
#
#     - The two disease nodes (T2DM, Melanoma)
#     - The two main pathway nodes (hsa04930, hsa05218)
#     - All shared pathway genes (yellow) — the 9 PI3K/MAPK axis genes
#     - All cross-disease genes (white) — AKT2 and equivalents
#     - Their immediate neighbours (1-hop) — genes, drugs, and pathways
#       directly connected to the special genes
#     - The Cell cycle and p53 signalling pathway nodes
#       (shared related pathways identified in the literature review)
#
#   Node size in this subgraph reflects BETWEENNESS CENTRALITY rather than
#   degree, so structurally important bridge nodes (AKT2, TP53, INS, IRS1)
#   appear larger — making the mechanistic bottlenecks immediately visible.

#   Edge relation type labels are shown directly on edge lines (not just
#   on hover) since the subgraph has few enough edges to support this
#   without creating clutter.

# INPUT:  G, G_undirected, centrality_df, shared_pathway_genes,
#         cross_disease_genes (all from previous cells)
# OUTPUT: kg_poster_v1.html — focused subgraph, opens in any browser




from pyvis.network import Network

# ── Step 1: Define the node set for the subgraph ─────────────────────────────
# Start with the anchor nodes and special gene categories, then expand
# by one hop to include their immediate neighbours. This captures the
# local biological context of each important node without including the
# entire graph.

# Anchor nodes — use seed nodes only (not KEGG disease nodes)
# Each disease has two nodes: seed (Disease:T2DM) and KEGG (KEGG:Disease:H00409)
# We include only the seed nodes here to avoid duplicate disease entries.
# The KEGG pathway nodes are included since they have no duplicate.
# Anchor nodes — always included regardless of connectivity

anchor_nodes = {
    "Disease:T2DM",
    "Disease:Melanoma",
    "KEGG:Disease:H00409",
    "KEGG:Disease:H00038",
    "KEGG:Pathway:hsa04930",
    "KEGG:Pathway:hsa05218",
}

shared_related_pathways = {
    n for n in G.nodes
    if G.nodes[n].get("node_type") == "Pathway"
    and any(name in G.nodes[n].get("name", "")
            for name in ["Cell cycle", "p53 signaling", "p53 signalling"])
}

special_genes = shared_pathway_genes | cross_disease_genes

one_hop_neighbours = set()
for gene in special_genes:
    for neighbour in G_undirected.neighbors(gene):
        edges_between = (G.get_edge_data(gene, neighbour) or
                         G.get_edge_data(neighbour, gene))
        if edges_between:
            if edges_between.get("relation", "") != "INTERACTS_WITH":
                one_hop_neighbours.add(neighbour)

top_centrality_genes = set(
    centrality_df[centrality_df["node_type"] == "Gene"]
    .head(10)["node_id"].tolist()
)

subgraph_nodes = (
    anchor_nodes | special_genes |
    shared_related_pathways |
    one_hop_neighbours | top_centrality_genes
)
subgraph_nodes = {n for n in subgraph_nodes if n in G.nodes}

print(f"Subgraph nodes: {len(subgraph_nodes)}")

# Build the subgraph.
# Extract the induced subgraph — only edges where BOTH endpoints are in
# our selected node set. This preserves all relationships between
# selected nodes without including edges to excluded nodes.

H = G.subgraph(subgraph_nodes).copy()
print(f"Subgraph edges: {H.number_of_edges()}")
print()


# ── Step 2: Colour maps — adjusted for WHITE background ───────────────────────

NODE_COLOR_MAP = {
    "Disease":    "#e63946",
    "Pathway":    "#e07b00",
    "Drug":       "#1d6fa4",
    "Gene":       "#2d8a4e",
    "Metabolite": "#7b2fa8",
}
SHARED_COLOR        = "#c9a800"   # dark yellow
CROSS_DISEASE_COLOR = "#1a1a6e"   # dark navy — replaces white
DEFAULT_NODE_COLOR  = "#666666"

# Edge label colours — must be dark and readable on white background
EDGE_LABEL_COLORS = {
    "HAS_GENE":         "#e07b00",
    "HAS_DRUG":         "#1d6fa4",
    "TARGETS":          "#e63946",
    "RELATED_PATHWAY":  "#c9a800",
    "HAS_METABOLITE":   "#7b2fa8",
    "SAME_AS":          "#999999",
    "HAS_KEGG_PATHWAY": "#00897b",
    "INTERACTS_WITH":   "#cccccc",
}


# ── Step 3: Node sizing by betweenness centrality ─────────────────────────────
# In the poster subgraph, node size reflects betweenness centrality
# rather than degree. This makes structural bridge nodes (AKT2, TP53,
# INS, IRS1) visually prominent.

bc_scores   = {n: G.nodes[n].get("betweenness", 0) for n in H.nodes}
max_bc      = max(bc_scores.values()) if bc_scores else 1
MIN_SIZE    = 15
MAX_SIZE    = 75
MIN_SPECIAL = 30

def scale_by_centrality(node_id, is_special=False):
    bc     = bc_scores.get(node_id, 0)
    scaled = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * (bc / max_bc)
    return max(scaled, MIN_SPECIAL) if is_special else max(scaled, MIN_SIZE)


# ── Step 4: Initialise network — WHITE background ─────────────────────────────
net = Network(
    height     = "950px",
    width      = "100%",
    bgcolor    = "#ffffff",    # white background
    font_color = "#222222",    # dark labels
    directed   = True,
)

net.barnes_hut(
    gravity         = -15000,
    central_gravity = 0.15,
    spring_length   = 250,
    spring_strength = 0.03,
    damping         = 0.09,
)


# ── Step 5: Add nodes ─────────────────────────────────────────────────────────
for node_id, attr in H.nodes(data=True):
    ntype  = attr.get("node_type", "Gene")
    name   = attr.get("name", node_id)
    notes  = str(attr.get("notes", "") or "")
    source = str(attr.get("source_db", "") or "")
    ref    = str(attr.get("source_ref", "") or "")
    bc     = attr.get("betweenness", 0)
    deg    = G_undirected.degree(node_id)

    is_special = False
    if node_id in cross_disease_genes:
        color      = CROSS_DISEASE_COLOR
        is_special = True
        category   = "Cross-disease gene (T2DM disease ↔ Melanoma pathway)"
    elif node_id in shared_pathway_genes or "Shared" in notes:
        color      = SHARED_COLOR
        is_special = True
        category   = "Shared pathway gene (hsa04930 ∩ hsa05218)"
    else:
        color    = NODE_COLOR_MAP.get(ntype, DEFAULT_NODE_COLOR)
        category = ntype

    tooltip = (
        f"{name}\n"
        f"Category: {category}\n"
        f"Node type: {ntype}\n"
        f"Betweenness centrality: {bc:.6f}\n"
        f"Degree (full graph): {deg}\n"
        f"Source: {source}\n"
        f"Notes: {notes}"
    )
    
    if ref:
        tooltip += f"\nRef: {ref}"

    net.add_node(
        node_id,
        label = name,
        color = color,
        size  = scale_by_centrality(node_id, is_special),
        title = tooltip,
        font  = {"color": "#222222", "size": 11, "face": "Arial"},
    )



# ── Step 6: Add edges with labels shown directly on edge lines ────────────────
# The poster subgraph has few enough edges that permanent edge labels
# are readable. Labels are shown in the same colour as the edge so they
# visually belong to that edge type. INTERACTS_WITH edges are not labelled
# to avoid clutter.

for u, v, attr in H.edges(data=True):
    relation = attr.get("relation", "")
    source   = attr.get("source_db", "")
    notes    = str(attr.get("notes", "") or "")

    edge_color   = EDGE_LABEL_COLORS.get(relation, "#aaaaaa")
    edge_tooltip = (
        f"Relation: {relation}\n"
        f"Source: {source}\n"
        f"Notes: {notes}"
    )

    if relation == "INTERACTS_WITH":
        # PPI edges — no label, very faint
        net.add_edge(u, v,
                     title = edge_tooltip,
                     color = "rgba(180,180,180,0.3)",
                     width = 0.5,
                     label = "")
    else:
        # All other edges — label shown directly on the edge line
        net.add_edge(u, v,
                     title  = edge_tooltip,
                     color  = edge_color,
                     width  = 2.0,
                     label  = relation,   # shown permanently on edge line
                     font   = {
                         "size":        9,
                         "color":       edge_color,
                         "strokeWidth": 3,
                         "strokeColor": "#ffffff",  # white stroke makes label
                         "align":       "middle",   # readable on white bg
                     })


# ── Step 7: Save and patch ────────────────────────────────────────────────────
output_file = "kg_poster_v1.html"
net.write_html(output_file)

# Patch 1: Enable edge hover tooltips
import re

with open(output_file, "r", encoding="utf-8") as f:
    html = f.read()

if "hoverEdge" in html:
    html = re.sub(
        r'/\* KG_TOOLTIP_START \*/.+?/\* KG_TOOLTIP_END \*/',
        '', html, flags=re.DOTALL
    )

# Patch 2: Set white background explicitly in body style
# (ensures white even if browser default differs)
html = html.replace(
    "<body>",
    "<body style='background-color:#ffffff;'>"
)

# # Patch 3: Edge tooltip handler
# edge_tooltip_js = """
# <style>
# #kg-poster-tip {
#     position: fixed;
#     background: rgba(255,255,255,0.97);
#     color: #222222;
#     padding: 9px 14px;
#     border-radius: 7px;
#     font-size: 13px;
#     font-family: Arial, sans-serif;
#     border: 1.5px solid #e07b00;
#     pointer-events: none;
#     z-index: 99999;
#     display: none;
#     max-width: 340px;
#     line-height: 1.7;
#     box-shadow: 0 3px 12px rgba(0,0,0,0.15);
# }
# </style>
# <div id="kg-poster-tip"></div>
# <script>
# /* KG_TOOLTIP_START */
# var tooltipPoll = setInterval(function() {
#     if (typeof network !== "undefined") {
#         clearInterval(tooltipPoll);
#         var tip = document.getElementById("kg-poster-tip");

#         network.setOptions({ interaction: { hover: true, tooltipDelay: 50 } });

#         network.on("hoverEdge", function(params) {
#             var edge = network.body.data.edges.get(params.edge);
#             if (!edge) return;
#             var content = edge.title || edge.label || ("Edge: " + params.edge);
#             tip.innerHTML = content.replace(/\\n/g, "<br>");
#             tip.style.display = "block";
#         });

#         network.on("blurEdge", function() { tip.style.display = "none"; });
#         network.on("click",    function() { tip.style.display = "none"; });

#         document.addEventListener("mousemove", function(e) {
#             if (tip.style.display === "block") {
#                 tip.style.left = (e.clientX + 16) + "px";
#                 tip.style.top  = (e.clientY - 10) + "px";
#             }
#         });
#     }
# }, 100);
# /* KG_TOOLTIP_END */
# </script>"""

# html = html.replace("</body>", edge_tooltip_js + "\n</body>") \
#        if "</body>" in html else html + edge_tooltip_js

with open(output_file, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved: {output_file}")
print()
print("=== POSTER SUBGRAPH SUMMARY ===")
from collections import Counter
type_counts = Counter(H.nodes[n]["node_type"] for n in H.nodes)
for ntype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ntype:<12} {count}")
print()

print("Cell 8 complete. Proceed to Cell 8a.")

Subgraph nodes: 27
Subgraph edges: 92

Saved: kg_poster_v1.html

=== POSTER SUBGRAPH SUMMARY ===
  Gene         19
  Pathway      4
  Disease      4

Cell 8 complete. Proceed to Cell 8a.


In [40]:
# =============================================================================
# CELL 8a — Patch kg_poster_v1.html with working edge tooltips
#
# PURPOSE:
#   The hoverEdge event in vis.js only fires when interaction.hover = true.
#   PyVis does not enable this by default. This cell reads the poster HTML,
#   finds the vis.js Network instantiation line, and injects two things
#   immediately after it:
#     1. network.setOptions({interaction:{hover:true}}) — enables hover events
#     2. The hoverEdge event listener — shows the edge tooltip
#   This is the minimal targeted fix — it does not touch anything else.
# =============================================================================

import re

poster_file = "kg_poster_v1.html"

# Read the poster HTML
with open(poster_file, "r", encoding="utf-8") as f:
    html = f.read()

# ── Step 1: Check if already patched ─────────────────────────────────────────
if "hoverEdge" in html:
    print("File already contains hoverEdge handler — removing old injection first.")
    # Remove everything between our injection markers if they exist
    html = re.sub(
        r'/\* KG_TOOLTIP_START \*/.+?/\* KG_TOOLTIP_END \*/',
        '',
        html,
        flags=re.DOTALL
    )

# ── Step 2: Find the network instantiation line ───────────────────────────────
# PyVis always creates the network with this exact pattern:
#   network = new vis.Network(container, data, options);
# We find it and inject our code immediately after the semicolon.

pattern = r'(network\s*=\s*new\s+vis\.Network\([^;]+;)'
match   = re.search(pattern, html)

if not match:
    print("ERROR: Could not find 'network = new vis.Network(...)' in the HTML.")
    print("The file structure may be different. Check the file manually.")
else:
    print(f"Found network instantiation at character position {match.start()}")

    # The code to inject — placed immediately after network creation
    injection = """
/* KG_TOOLTIP_START */
/* Enable hover events — required for hoverEdge to fire in vis.js */
network.setOptions({ interaction: { hover: true, tooltipDelay: 50 } });

/* Create the tooltip element */
var _kgTip = document.createElement("div");
_kgTip.id = "kg-poster-tip";
_kgTip.style.cssText = [
    "position:fixed",
    "background:rgba(10,15,35,0.96)",
    "color:#f0f0f0",
    "padding:9px 14px",
    "border-radius:7px",
    "font-size:13px",
    "font-family:Arial,sans-serif",
    "border:1.5px solid #ffa94d",
    "pointer-events:none",
    "z-index:99999",
    "display:none",
    "max-width:340px",
    "line-height:1.7",
    "box-shadow:0 3px 12px rgba(0,0,0,0.6)"
].join(";");
document.body.appendChild(_kgTip);

/* hoverEdge — fires when cursor is over an edge */
network.on("hoverEdge", function(params) {
    var edge = network.body.data.edges.get(params.edge);
    if (!edge) return;
    var content = edge.title || edge.label || ("Edge: " + params.edge);
    _kgTip.innerHTML = content.replace(/\\n/g, "<br>");
    _kgTip.style.display = "block";
});

/* blurEdge — fires when cursor leaves an edge */
network.on("blurEdge", function() {
    _kgTip.style.display = "none";
});

/* Track mouse position to keep tooltip near cursor */
document.addEventListener("mousemove", function(e) {
    if (_kgTip.style.display === "block") {
        _kgTip.style.left = (e.clientX + 16) + "px";
        _kgTip.style.top  = (e.clientY - 10) + "px";
    }
});

/* Hide on canvas click */
network.on("click", function() { _kgTip.style.display = "none"; });
/* KG_TOOLTIP_END */"""

    # Insert injection after the network instantiation line
    original_line = match.group(0)
    html = html.replace(original_line, original_line + injection, 1)

    # Write patched HTML back to disk
    with open(poster_file, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Patched successfully: {poster_file}")

File already contains hoverEdge handler — removing old injection first.
Found network instantiation at character position 40641
Patched successfully: kg_poster_v1.html


In [80]:
# # CELL 9 — Static high-quality image export (PNG and SVG)
# #
# # PURPOSE:
# #   Generate publication-quality static images of the knowledge graph for
# #   use in the dissertation and poster. Unlike the interactive HTML files,
# #   static images can be embedded directly into Word documents, PowerPoint
# #   slides, and PDF posters without requiring a browser.
# #
# #   Two outputs are produced:
# #     1. PNG at 300 DPI — suitable for print (dissertation figures, poster)
# #     2. SVG (vector format) — infinitely scalable, ideal for poster printing
# #        at large sizes without pixellation
# #
# #   The static image uses matplotlib with a spring layout from NetworkX.
# #   Because matplotlib cannot replicate the exact physics-based layout of
# #   the PyVis graph, we use the poster subgraph (H) with a fixed random
# #   seed so the layout is reproducible.
# #
# #   Node colours, sizes, and categories match the interactive graph exactly
# #   so both outputs are visually consistent.
# #
# # INPUT:  H (poster subgraph from Cell 7), centrality scores,
# #         shared_pathway_genes, cross_disease_genes
# # OUTPUT: kg_poster_static.png  — 300 DPI raster image
# #         kg_poster_static.svg  — vector image (scalable)


# import matplotlib
# matplotlib.use("Agg")        # non-interactive backend — required for saving files
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches   # for the legend
# import networkx as nx
# import numpy as np

# print("Generating static poster image...")
# print()

# # ── Step 1: Compute layout ────────────────────────────────────────────────────
# # spring_layout (Fruchterman-Reingold) produces a force-directed layout
# # similar in character to the Barnes-Hut physics in PyVis.
# # seed=42 ensures the layout is identical every time this cell is run,
# # which is essential for a reproducible dissertation figure.
# # k controls the optimal distance between nodes — higher = more spread out.

# pos = nx.spring_layout(
#     H,
#     seed = 42,      # fixed seed for reproducibility
#     k    = 2.5,     # node spacing (higher = more spread)
# )


# # ── Step 2: Prepare node colours ──────────────────────────────────────────────
# # Colours match the white-background HTML visualisations.
# # Cross-disease gene (AKT2) uses dark navy (#1a1a6e)
# # because white is invisible on the white background of the static image.

# NODE_COLOR_MAP_STATIC = {
#     "Disease":    "#e63946",   # dark red
#     "Pathway":    "#e07b00",   # dark orange
#     "Drug":       "#1d6fa4",   # dark blue
#     "Gene":       "#2d8a4e",   # dark green
#     "Metabolite": "#7b2fa8",   # dark purple
# }
# SHARED_COLOR_STATIC        = "#c9a800"   # dark yellow
# CROSS_DISEASE_COLOR_STATIC = "#1a1a6e"   # dark navy — AKT2
# DEFAULT_COLOR_STATIC       = "#666666"

# node_colors = []
# node_sizes  = []
# max_bc      = max(
#     (H.nodes[n].get("betweenness", 0) for n in H.nodes), default=1
# )

# for node_id in H.nodes:
#     attr  = H.nodes[node_id]
#     ntype = attr.get("node_type", "Gene")
#     notes = str(attr.get("notes", "") or "")
#     bc    = attr.get("betweenness", 0)

#     # Colour priority: cross-disease > shared > type
#     if node_id in cross_disease_genes:
#         color = CROSS_DISEASE_COLOR_STATIC   # dark navy for AKT2
#     elif node_id in shared_pathway_genes or "Shared" in notes:
#         color = SHARED_COLOR_STATIC
#     else:
#         color = NODE_COLOR_MAP_STATIC.get(ntype, DEFAULT_COLOR_STATIC)

#     node_colors.append(color)

#     # Size by betweenness centrality
#     scaled = 200 + 2800 * (bc / max_bc) if max_bc > 0 else 200
#     if node_id in cross_disease_genes or node_id in shared_pathway_genes:
#         scaled = max(scaled, 600)
#     node_sizes.append(scaled)

# # ── Step 3: Prepare edge colours ─────────────────────────────────────────────
# # Match the edge colour scheme from the interactive graph

# EDGE_COLOR_MAP_STATIC = {
#     "HAS_GENE":         "#ffa94d",
#     "HAS_DRUG":         "#63b3ed",
#     "TARGETS":          "#ff4757",
#     "INTERACTS_WITH":   "#333333",
#     "RELATED_PATHWAY":  "#f6c90e",
#     "HAS_METABOLITE":   "#da77f2",
#     "SAME_AS":          "#555555",
#     "HAS_KEGG_PATHWAY": "#63e6be",
# }

# edge_colors = []
# edge_widths = []

# for u, v, attr in H.edges(data=True):
#     relation = attr.get("relation", "")
#     color    = EDGE_COLOR_MAP_STATIC.get(relation, "#555555")
#     width    = 0.4 if relation == "INTERACTS_WITH" else 1.2
#     edge_colors.append(color)
#     edge_widths.append(width)


# # ── Step 4: Prepare node labels ───────────────────────────────────────────────
# # Only label nodes that are special (shared genes, cross-disease, disease,
# # pathway) or have high betweenness — labelling every node creates clutter.
# # Threshold: label nodes with betweenness > 0.01 OR that are special

# label_threshold = 0.01

# labels = {}
# for node_id in H.nodes:
#     attr  = H.nodes[node_id]
#     bc    = attr.get("betweenness", 0)
#     ntype = attr.get("node_type", "")
#     name  = attr.get("name", node_id)
#     notes = str(attr.get("notes", "") or "")

#     should_label = (
#         node_id in cross_disease_genes or
#         node_id in shared_pathway_genes or
#         ntype in ("Disease", "Pathway") or
#         bc >= label_threshold
#     )
#     if should_label:
#         # Truncate long names for readability
#         labels[node_id] = name[:22] + "…" if len(name) > 22 else name


# # ── Step 5: Draw the graph ────────────────────────────────────────────────────

# fig, ax = plt.subplots(figsize=(20, 16)) # large figure for high-resolution output
# ax.set_facecolor("#ffffff")          # white background
# fig.patch.set_facecolor("#ffffff")   # white figure background

# # Node colours for white background
# NODE_COLOR_MAP = {
#     "Disease":    "#e63946",
#     "Pathway":    "#e07b00",
#     "Drug":       "#1d6fa4",
#     "Gene":       "#2d8a4e",
#     "Metabolite": "#7b2fa8",
# }
# SHARED_COLOR        = "#c9a800"
# CROSS_DISEASE_COLOR = "#1a1a6e"   # dark navy replaces white
# DEFAULT_COLOR       = "#666666"

# # Draw edges first (behind nodes)
# nx.draw_networkx_edges(
#     H, pos,
#     ax          = ax,
#     edge_color  = edge_colors,
#     width       = edge_widths,
#     alpha       = 0.7,
#     arrows      = True,
#     arrowsize   = 10,
#     arrowstyle  = "-|>",
#     node_size   = node_sizes,   # needed so arrows terminate at node edge, not centre
# )

# # Show labels for non-PPI edges only, to avoid clutter

# edge_labels_by_relation = {}

# for u, v, attr in H.edges(data=True):
#     relation = attr.get("relation", "")
#     if relation and relation != "INTERACTS_WITH":
#         edge_labels_by_relation.setdefault(relation, {})
#         edge_labels_by_relation[relation][(u, v)] = relation

# # Draw labels relation-by-relation so each relation can have its own colour
# for relation, rel_edge_labels in edge_labels_by_relation.items():
#     label_color = EDGE_COLOR_MAP_STATIC.get(relation, "#555555")

#     nx.draw_networkx_edge_labels(
#         H,
#         pos,
#         edge_labels = rel_edge_labels,
#         ax          = ax,
#         font_size   = 5.5,
#         font_color  = label_color,
#         font_family = "Arial",
#         rotate      = False,
#         label_pos   = 0.5,
#         bbox        = dict(
#             facecolor = "#ffffff",
#             edgecolor = "none",
#             alpha     = 0.75,
#             pad       = 0.15,
#         ),
#     )

# # Draw nodes on top of edges
# nx.draw_networkx_nodes(
#     H, pos,
#     ax         = ax,
#     node_color = node_colors,
#     node_size  = node_sizes,
#     linewidths = 0.8,
#     edgecolors = "#ffffff33",   # faint white border around nodes
# )

# # Draw labels for important nodes only
# # Labels — dark text on white background
# nx.draw_networkx_labels(
#     H, pos,
#     labels      = labels,
#     ax          = ax,
#     font_size   = 7,
#     font_color  = "#222222",    # dark text
#     font_family = "Arial",
# )

# # ── Step 6: Add legend ────────────────────────────────────────────────────────
# # Build a compact legend matching the node and edge colour scheme


# legend_elements = [
#     # Node types
#     mpatches.Patch(color="#e63946", label="Disease"),
#     mpatches.Patch(color="#e07b00", label="Pathway"),
#     mpatches.Patch(color="#1d6fa4", label="Drug"),
#     mpatches.Patch(color="#2d8a4e", label="Gene"),
#     mpatches.Patch(color="#7b2fa8", label="Metabolite"),
#     mpatches.Patch(color="#c9a800", label="Shared pathway gene"),
#     mpatches.Patch(color="#1a1a6e", label="Cross-disease gene (AKT2)"),
# ]

# legend = ax.legend(
#     handles        = legend_elements,
#     loc            = "upper left",
#     fontsize       = 9,
#     framealpha     = 0.3,
#     facecolor      = "#06101a",
#     edgecolor      = "#333355",
#     labelcolor     = "white",
#     title          = "Node Types",
#     title_fontsize = 9,
# )
# legend.get_title().set_color("white")

# # Title and axis cleanup
# # Labels — dark text on white background
# ax.set_title(
#     "T2DM–Melanoma Knowledge Graph — Shared Molecular Interface",
#     color    = "#222222",   # dark title
#     fontsize = 14,
#     pad      = 15,
# )


# # ── Step 7: Save PNG and SVG ──────────────────────────────────────────────────
# # PNG at 300 DPI — standard for print-quality dissertation figures

# plt.savefig("kg_poster_static.png", dpi=300, bbox_inches="tight",
#             facecolor="#ffffff")
# print("Saved: kg_poster_static.png (300 DPI)")

# # SVG — vector format, infinitely scalable for large poster printing
# plt.savefig("kg_poster_static.svg", format="svg", bbox_inches="tight",
#             facecolor="#ffffff")
# print("Saved: kg_poster_static.svg (vector, scalable)")

# plt.close()   # free memory

# print()
# print("=== IMAGE SUMMARY ===")
# print()
# print("Cell 9 complete. Proceed to Cell 10: export final results to Excel.")

Generating static poster image...

Saved: kg_poster_static.png (300 DPI)
Saved: kg_poster_static.svg (vector, scalable)

=== IMAGE SUMMARY ===

Cell 9 complete. Proceed to Cell 10: export final results to Excel.


In [26]:
# CELL 9 — Static high-quality image export (PNG and SVG)
#
# PURPOSE:
#   Generate a publication-quality static image of the poster subgraph for
#   use in the dissertation. Unlike the interactive HTML, this can be embedded
#   directly into Word documents and PDF submissions.
#
# CHANGES FROM PREVIOUS VERSION:
#   - Font size increased to 14pt for node labels
#   - Label density reduced — only disease, pathway, AKT2, shared genes,
#     and top-BC genes (BC ≥ 0.06) are labelled to prevent clutter
#   - k increased to 4.0 so nodes spread further apart, reducing overlap
#   - INTERACTS_WITH edges now visible in medium grey (was too faint)
#   - Edge relationship labels drawn directly on non-PPI edges
#   - Node sizes scaled more aggressively for clearer visual hierarchy
#   - Saved as both PNG (300 DPI) and SVG (infinitely scalable vector)
#
# INPUT:  H (poster subgraph from Cell 8), cross_disease_genes,
#         shared_pathway_genes (from Cell 4)
# OUTPUT: fig3_kg_subgraph_static.png — 300 DPI raster image
#         fig3_kg_subgraph_static.svg — vector image for poster printing

import matplotlib
matplotlib.use("Agg")        # non-interactive backend — required for file saving
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches   # for building the legend
import networkx as nx
import numpy as np

print("Generating static poster image...")
print()


# ── Step 1: Compute spring layout ─────────────────────────────────────────────
# spring_layout (Fruchterman-Reingold) produces a force-directed layout
# similar in character to the Barnes-Hut physics used in the PyVis HTML graph.
# seed=42 ensures the layout is identical every time this cell is run —
# essential for a reproducible dissertation figure.
# k=4.0 sets the optimal distance between nodes; higher values push nodes
# further apart, which is the primary fix for overlapping labels.

pos = nx.spring_layout(
    H,
    seed = 42,    # fixed seed for layout reproducibility
    k    = 4.0,   # increased from 2.5 — more node spacing, prevents label overlap
)


# ── Step 2: Define colour maps for white background ───────────────────────────
# Colours match the interactive HTML graph for visual consistency.
# Cross-disease gene (AKT2) uses dark navy (#1a1a6e) rather than white,
# because white nodes are invisible on a white background.

NODE_COLOR_MAP_STATIC = {
    "Disease":    "#e63946",   # dark red    — disease anchor nodes
    "Pathway":    "#e07b00",   # dark orange — KEGG pathway nodes
    "Drug":       "#1d6fa4",   # dark blue   — drug nodes
    "Gene":       "#2d8a4e",   # dark green  — standard gene nodes
    "Metabolite": "#7b2fa8",   # dark purple — metabolite nodes
}
SHARED_COLOR_STATIC        = "#c9a800"   # dark yellow — shared PI3K/MAPK pathway genes
CROSS_DISEASE_COLOR_STATIC = "#1a1a6e"   # dark navy   — AKT2 (cross-disease gene)
DEFAULT_COLOR_STATIC       = "#666666"   # mid grey    — fallback

# Edge colours keyed by relation type — each relationship has a distinct colour
# to match the interactive graph colour scheme
EDGE_COLOR_MAP_STATIC = {
    "HAS_GENE":         "#e07b00",   # orange — pathway/disease to gene
    "HAS_DRUG":         "#1d6fa4",   # blue   — pathway/disease to drug
    "TARGETS":          "#e63946",   # red    — drug to gene target
    "INTERACTS_WITH":   "#aaaaaa",   # grey   — STRING PPI (subordinate but visible)
    "RELATED_PATHWAY":  "#c9a800",   # yellow — disease to related pathway
    "HAS_METABOLITE":   "#7b2fa8",   # purple — pathway to metabolite
    "SAME_AS":          "#888888",   # grey   — seed node to KEGG entry
    "HAS_KEGG_PATHWAY": "#00897b",   # teal   — disease to main pathway
}


# ── Step 3: Build node colour, size, and label lists in ONE loop ───────────────
# IMPORTANT: node_colors, node_sizes, and priority_labels must all be built
# in a single pass over H.nodes to ensure list lengths match the node count.
# Having two separate loops was the cause of the previous ValueError —
# it doubled the list lengths, making them inconsistent with the node count.

max_bc = max(
    (H.nodes[n].get("betweenness", 0) for n in H.nodes), default=1
)   # normalisation denominator for size scaling

node_colors    = []   # one colour per node, in iteration order
node_sizes     = []   # one size per node, in iteration order
priority_labels = {}  # {node_id: label_string} — only for key nodes

for node_id in H.nodes:
    attr  = H.nodes[node_id]
    ntype = attr.get("node_type", "Gene")
    name  = attr.get("name", node_id)
    notes = str(attr.get("notes", "") or "")
    bc    = attr.get("betweenness", 0)

    # ── Colour assignment — priority: cross-disease > shared > type-based ──
    if node_id in cross_disease_genes:
        color = CROSS_DISEASE_COLOR_STATIC   # dark navy for AKT2
    elif node_id in shared_pathway_genes or "Shared" in notes:
        color = SHARED_COLOR_STATIC          # dark yellow for shared PI3K/MAPK genes
    else:
        color = NODE_COLOR_MAP_STATIC.get(ntype, DEFAULT_COLOR_STATIC)

    node_colors.append(color)

    # ── Size assignment — scaled by betweenness centrality ────────────────
    # Minimum 800, maximum 6000 (for the highest-BC disease/pathway nodes).
    # Special nodes (AKT2, shared genes) guaranteed a minimum of 1200
    # so they are always clearly visible regardless of their BC score.
    scaled = 800 + 5200 * (bc / max_bc) if max_bc > 0 else 800
    if node_id in cross_disease_genes or node_id in shared_pathway_genes:
        scaled = max(scaled, 1200)
    node_sizes.append(scaled)

    # ── Label assignment — key nodes only (per supervisor feedback) ────────
    # We label only: disease nodes, pathway nodes, AKT2 (cross-disease gene),
    # shared pathway genes, and top-BC genes (BC ≥ 0.06, i.e. AKT2, TP53,
    # INS, IRS1). All other nodes are shown without labels to prevent clutter
    # and overlapping text in the dense gene region of the subgraph.
    should_label = (
        node_id in cross_disease_genes or       # AKT2 — always label
        node_id in shared_pathway_genes or      # 9 shared PI3K/MAPK genes
        ntype in ("Disease", "Pathway") or      # all disease and pathway nodes
        bc >= 0.06                               # top structural bridge genes only
    )
    if should_label:
        # Truncate very long pathway names to prevent label overflow
        label = name[:20] + "…" if len(name) > 20 else name
        priority_labels[node_id] = label

print(f"Nodes labelled: {len(priority_labels)} of {H.number_of_nodes()}")
print(f"Labelled nodes: {list(priority_labels.values())}")
print()


# ── Step 4: Prepare edge colours and widths ───────────────────────────────────
# PPI (INTERACTS_WITH) edges use a lighter grey and thinner line so they
# appear as background texture rather than foreground structure.
# All other edges are drawn thicker and in their relation-type colour.

edge_colors = []
edge_widths = []
for u, v, attr in H.edges(data=True):
    rel = attr.get("relation", "")
    edge_colors.append(EDGE_COLOR_MAP_STATIC.get(rel, "#aaaaaa"))
    edge_widths.append(1.0 if rel == "INTERACTS_WITH" else 2.5)


# ── Step 5: Prepare edge relationship labels ───────────────────────────────────
# Build a dict of {(source, target): relation_type} for all non-PPI edges.
# INTERACTS_WITH edges are excluded because there are many of them and
# labelling all would create unreadable clutter in the graph.

edge_labels = {
    (u, v): attr.get("relation", "")
    for u, v, attr in H.edges(data=True)
    if attr.get("relation", "") != "INTERACTS_WITH"
}


# ── Step 6: Draw the graph ────────────────────────────────────────────────────
# figsize=(22, 18) produces a large canvas for high-resolution output.
# All drawing is done in order: edges first (behind), then nodes, then labels.

fig, ax = plt.subplots(figsize=(22, 18))
ax.set_facecolor("#ffffff")          # white axes background
fig.patch.set_facecolor("#ffffff")   # white figure background

# Draw edges — behind all nodes
nx.draw_networkx_edges(
    H, pos, ax=ax,
    edge_color = edge_colors,
    width      = edge_widths,
    alpha      = 0.75,          # slightly transparent for visual clarity
    arrows     = True,          # show directional arrows
    arrowsize  = 20,            # increased from 14 — more visible at high DPI
    arrowstyle = "-|>",         # filled arrowhead style
    node_size  = node_sizes,    # needed so arrows terminate at node edge, not centre
)

# Draw nodes — on top of edges
nx.draw_networkx_nodes(
    H, pos, ax=ax,
    node_color = node_colors,   # type-based colours defined in Step 3
    node_size  = node_sizes,    # BC-scaled sizes defined in Step 3
    linewidths = 1.2,           # border thickness
    edgecolors = "#ffffff",     # white border separates overlapping nodes
)

# Draw node labels — key nodes only
nx.draw_networkx_labels(
    H, pos,
    labels      = priority_labels,   # restricted label set from Step 3
    ax          = ax,
    font_size   = 14,                # increased from 7 — substantially larger
    font_color  = "#111111",         # near-black for maximum contrast on white
    font_family = "Arial",
    font_weight = "bold",
)

# Draw edge relationship labels directly on edge lines
# Only non-PPI edges are labelled (INTERACTS_WITH excluded)
nx.draw_networkx_edge_labels(
    H, pos,
    edge_labels = edge_labels,    # {(u,v): relation_type} from Step 5
    ax          = ax,
    font_size   = 8,              # small but readable at 300 DPI
    font_color  = "#333333",
    font_family = "Arial",
    bbox        = dict(boxstyle="round,pad=0.2", facecolor="white",
                       edgecolor="none", alpha=0.7),   # white background prevents
                                                        # labels merging with edges
    label_pos   = 0.5,            # place label at midpoint of each edge
    rotate      = True,           # rotate label to follow edge direction
)


# ── Step 7: Add legend ─────────────────────────────────────────────────────────
# Compact legend matching the node colour scheme, positioned top-left
# to avoid obscuring the main graph structure.

legend_elements = [
    mpatches.Patch(color="#e63946", label="Disease"),
    mpatches.Patch(color="#e07b00", label="Pathway"),
    mpatches.Patch(color="#2d8a4e", label="Gene"),
    mpatches.Patch(color="#c9a800", label="Shared pathway gene"),
    mpatches.Patch(color="#1a1a6e", label="Cross-disease gene (AKT2)"),
]
legend = ax.legend(
    handles        = legend_elements,
    loc            = "upper left",
    fontsize       = 12,
    framealpha     = 0.9,
    facecolor      = "white",
    edgecolor      = "#cccccc",
    title          = "Node types",
    title_fontsize = 12,
)
legend.get_title().set_color("#222222")

# Figure title — format per supervisor feedback: no colon, left-aligned
ax.set_title(
    "Fig 3 Subgraph of the knowledge graph highlighting key biological features",
    color    = "#222222",
    fontsize = 14,
    pad      = 16,
    loc      = "left",
)
ax.axis("off")   # hide axis lines and tick marks
plt.tight_layout()


# ── Step 8: Save outputs ───────────────────────────────────────────────────────
# PNG at 300 DPI — standard for print-quality dissertation figures (Word, PDF)
# SVG — vector format, infinitely scalable for large poster printing

plt.savefig("fig3_kg_subgraph_static.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.savefig("fig3_kg_subgraph_static.svg", format="svg", bbox_inches="tight",
            facecolor="white")
plt.close()   # free memory

print("Saved: fig3_kg_subgraph_static.png (300 DPI)")
print("Saved: fig3_kg_subgraph_static.svg (vector)")

Generating static poster image...

Nodes labelled: 21 of 27
Labelled nodes: ['Type II diabetes mel…', 'PIK3R2', 'P3R3URF-PIK3R3', 'AKT2', 'T2DM', 'PIK3CB', 'PIK3R1', 'Melanoma', 'p53 signaling pathwa…', 'TP53', 'PIK3CD', 'Melanoma', 'PIK3R3', 'MAPK1', 'PIK3CA', 'T2DM', 'IRS1', 'Melanoma', 'INS', 'Cell cycle', 'MAPK3']

Saved: fig3_kg_subgraph_static.png (300 DPI)
Saved: fig3_kg_subgraph_static.svg (vector)


In [43]:
# CELL 10 — Export all analysis results to Excel
#
# PURPOSE:
#   Save all analysis outputs from Cells 6, 7, and 8 to a single Excel
#   workbook with one sheet per analysis. This produces a clean, shareable
#   record of all results that can be referenced in the dissertation and
#   submitted as supplementary data.
#
#   Sheets produced:
#     Sheet1 — Nodes          : full node table from the knowledge graph
#     Sheet2 — Edges          : full edge table from the knowledge graph
#     Sheet3 — Centrality     : all 290 nodes ranked by betweenness centrality
#     Sheet4 — DrugProximity  : all drugs ranked by proximity to melanoma
#     Sheet5 — SharedGenes    : the 9 shared pathway genes with their scores
#     Sheet6 — Summary        : key numerical findings for the results section
#
# INPUT:  nodes, edges, centrality_df, proximity_df, scored_df,
#         shared_pathway_genes, cross_disease_genes (from previous cells)
# OUTPUT: T2DM_Melanoma_KG_Results.xlsx


import pandas as pd
from openpyxl.styles import (Font, PatternFill, Alignment,
                              Border, Side)
from openpyxl import load_workbook

print("Exporting all results to Excel...")
print()

# ── Step 1: Prepare Sheet 3 — Betweenness centrality ─────────────────────────
# Full centrality table, already computed in Cell 5.
# Add a rank column for easy reference in the dissertation.

centrality_export = centrality_df[[
    "name", "node_type", "betweenness", "degree", "notes"
]].copy()
centrality_export.insert(0, "rank", range(1, len(centrality_export) + 1))
centrality_export.columns = [
    "Rank", "Name", "Node Type", "Betweenness Centrality", "Degree", "Notes"
]


# ── Step 2: Prepare Sheet 4 — Drug proximity scores ──────────────────────────
# Full drug proximity table from Cell 6, sorted by mean proximity.
# Include both scored drugs and unscored drugs (no targets) in one table
# with a Status column to distinguish them.

# Scored drugs — add rank
scored_export = scored_df[[
    "drug_name", "drug_class", "n_targets",
    "mean_proximity", "min_proximity", "status"
]].copy()
scored_export.insert(0, "rank", range(1, len(scored_export) + 1))

# Unscored drugs — no rank
# Add drug_class column before selecting — it was only added to scored_df
no_target_df["drug_class"] = no_target_df["drug_name"].apply(classify_drug)

unscored_export = no_target_df[[
    "drug_name", "drug_class", "n_targets",
    "mean_proximity", "min_proximity", "status"
]].copy()
unscored_export.insert(0, "rank", ["—"] * len(unscored_export))

# Combine scored and unscored into one table
drug_export = pd.concat(
    [scored_export, unscored_export], ignore_index=True
)
drug_export.columns = [
    "Rank", "Drug Name", "Drug Class", "N Targets",
    "Mean Proximity to Melanoma", "Min Proximity to Melanoma", "Status"
]


# ── Step 3: Prepare Sheet 5 — Special genes summary ──────────────────────────
# One table combining shared pathway genes and cross-disease genes
# with their centrality scores and drug proximity context.

special_gene_records = []

for gene_id in sorted(shared_pathway_genes | cross_disease_genes):
    if gene_id not in G.nodes:
        continue
    attr      = G.nodes[gene_id]
    name      = attr.get("name", gene_id)
    bc        = attr.get("betweenness", 0)
    degree    = G_undirected.degree(gene_id)
    notes     = str(attr.get("notes", "") or "")

    # Determine category
    if gene_id in cross_disease_genes:
        category = "Cross-disease gene"
    else:
        category = "Shared pathway gene"

    # Find centrality rank
    rank_row = centrality_df[centrality_df["node_id"] == gene_id]
    bc_rank  = int(rank_row.index[0]) + 1 if len(rank_row) > 0 else "—"

    # Count how many drugs target this gene directly
    targeting_drugs = [
        u for u, v, d in G.edges(data=True)
        if v == gene_id and d["relation"] == "TARGETS"
    ]

    special_gene_records.append({
        "Gene":               name,
        "Category":           category,
        "Betweenness Centrality": round(bc, 6),
        "Centrality Rank (all nodes)": bc_rank,
        "Degree":             degree,
        "Drugs Targeting Directly": len(targeting_drugs),
        "Notes":              notes,
    })

special_genes_df = pd.DataFrame(special_gene_records).sort_values(
    "Betweenness Centrality", ascending=False
).reset_index(drop=True)


# ── Step 4: Prepare Sheet 6 — Summary statistics ─────────────────────────────
# Key numerical findings formatted for easy copying into the dissertation.

summary_records = [
    # KG composition
    ("Knowledge Graph", "Total nodes",                  G.number_of_nodes()),
    ("Knowledge Graph", "Total edges (directed)",       G.number_of_edges()),
    ("Knowledge Graph", "Gene nodes",                   sum(1 for n in G.nodes if G.nodes[n]["node_type"] == "Gene")),
    ("Knowledge Graph", "Drug nodes",                   sum(1 for n in G.nodes if G.nodes[n]["node_type"] == "Drug")),
    ("Knowledge Graph", "Pathway nodes",                sum(1 for n in G.nodes if G.nodes[n]["node_type"] == "Pathway")),
    ("Knowledge Graph", "Metabolite nodes",             sum(1 for n in G.nodes if G.nodes[n]["node_type"] == "Metabolite")),
    ("Knowledge Graph", "Disease nodes",                sum(1 for n in G.nodes if G.nodes[n]["node_type"] == "Disease")),
    ("Knowledge Graph", "STRING PPI edges (≥0.9)",      sum(1 for u,v,d in G.edges(data=True) if d["relation"] == "INTERACTS_WITH")),
    ("Knowledge Graph", "KEGG HAS_GENE edges",          sum(1 for u,v,d in G.edges(data=True) if d["relation"] == "HAS_GENE")),
    ("Knowledge Graph", "Drug TARGETS edges",           sum(1 for u,v,d in G.edges(data=True) if d["relation"] == "TARGETS")),
    ("Knowledge Graph", "Connected components",         nx.number_connected_components(G_undirected)),
    # Shared genes
    ("Shared Genes",    "Shared pathway genes (hsa04930 ∩ hsa05218)", len(shared_pathway_genes)),
    ("Shared Genes",    "Cross-disease genes (disease entry ↔ pathway)", len(cross_disease_genes)),
    # Centrality
    ("Centrality",      "Highest betweenness node",     centrality_df.iloc[0]["name"]),
    ("Centrality",      "Highest betweenness score",    centrality_df.iloc[0]["betweenness"]),
    ("Centrality",      "Highest betweenness gene",     centrality_df[centrality_df["node_type"]=="Gene"].iloc[0]["name"]),
    ("Centrality",      "Highest gene betweenness score", centrality_df[centrality_df["node_type"]=="Gene"].iloc[0]["betweenness"]),
    ("Centrality",      "Nodes with betweenness > 0",   int((centrality_df["betweenness"] > 0).sum())),
    # Drug proximity
    ("Drug Proximity",  "Drugs with computed proximity scores", len(scored_df)),
    ("Drug Proximity",  "Drugs with no targets (unscored)",     len(no_target_df)),
    ("Drug Proximity",  "Closest drug class to melanoma",       class_summary.iloc[0]["drug_class"]),
    ("Drug Proximity",  "Closest drug class mean proximity",    class_summary.iloc[0]["mean_proximity"]),
    ("Drug Proximity",  "Most distant drug class",              class_summary.iloc[-1]["drug_class"]),
    ("Drug Proximity",  "Most distant drug class mean proximity", class_summary.iloc[-1]["mean_proximity"]),
]

summary_df = pd.DataFrame(
    summary_records,
    columns=["Category", "Metric", "Value"]
)


# ── Step 5: Write all sheets to Excel ─────────────────────────────────────────

output_file = "T2DM_Melanoma_KG_Results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # Sheet1 — full node table
    pd.DataFrame(nodes).to_excel(
        writer, sheet_name="Sheet1_Nodes", index=False
    )

    # Sheet2 — full edge table
    pd.DataFrame(edges).to_excel(
        writer, sheet_name="Sheet2_Edges", index=False
    )

    # Sheet3 — betweenness centrality
    centrality_export.to_excel(
        writer, sheet_name="Sheet3_Centrality", index=False
    )

    # Sheet4 — drug proximity
    drug_export.to_excel(
        writer, sheet_name="Sheet4_DrugProximity", index=False
    )

    # Sheet5 — special genes
    special_genes_df.to_excel(
        writer, sheet_name="Sheet5_SpecialGenes", index=False
    )

    # Sheet6 — summary statistics
    summary_df.to_excel(
        writer, sheet_name="Sheet6_Summary", index=False
    )


# ── Step 6: Apply basic formatting with openpyxl ─────────────────────────────
# Bold headers, auto-column widths, and frozen top row on every sheet

wb = load_workbook(output_file)

header_fill = PatternFill(
    start_color="1a1a4e",
    end_color="1a1a4e",
    fill_type="solid"
)
header_font  = Font(bold=True, color="FFFFFF", size=11)
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
thin_border  = Border(
    bottom=Side(style="thin", color="444488")
)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Bold and colour the header row
    for cell in ws[1]:
        cell.font      = header_font
        cell.fill      = header_fill
        cell.alignment = header_align
        cell.border    = thin_border

    # Freeze the top row so headers stay visible when scrolling
    ws.freeze_panes = "A2"

    # Auto-fit column widths based on content
    for col in ws.columns:
        max_len = 0
        col_letter = col[0].column_letter
        for cell in col:
            try:
                cell_len = len(str(cell.value)) if cell.value else 0
                max_len  = max(max_len, cell_len)
            except Exception:
                pass
        # Cap column width at 60 characters to prevent very wide columns
        ws.column_dimensions[col_letter].width = min(max_len + 4, 60)

wb.save(output_file)

print(f"Saved: {output_file}")
print()
print("=== SHEETS WRITTEN ===")
for sheet_name in wb.sheetnames:
    ws   = wb[sheet_name]
    rows = ws.max_row - 1   # subtract header row
    print(f"  {sheet_name:<30} {rows} data rows")
print()
print("Cell 10 complete.")
print()
print("=== ALL CELLS COMPLETE ===")
print("Files produced:")
print("  kg_full_v1.html              — full interactive graph (290 nodes)")
print("  kg_poster_v1.html            — poster subgraph (interactive)")
print("  kg_poster_static.png         — static poster image (300 DPI)")
print("  kg_poster_static.svg         — static poster image (vector)")
print("  T2DM_Melanoma_KG_Dataset_auto6.xlsx  — cleaned dataset (STRING 0.9)")
print("  T2DM_Melanoma_KG_Results.xlsx        — all analysis results")

Exporting all results to Excel...



C:\Users\USER\AppData\Local\Temp\ipykernel_26160\2013578191.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_target_df["drug_class"] = no_target_df["drug_name"].apply(classify_drug)


Saved: T2DM_Melanoma_KG_Results.xlsx

=== SHEETS WRITTEN ===
  Sheet1_Nodes                   290 data rows
  Sheet2_Edges                   1027 data rows
  Sheet3_Centrality              290 data rows
  Sheet4_DrugProximity           102 data rows
  Sheet5_SpecialGenes            10 data rows
  Sheet6_Summary                 24 data rows

Cell 10 complete.

=== ALL CELLS COMPLETE ===
Files produced:
  kg_full_v1.html              — full interactive graph (290 nodes)
  kg_poster_v1.html            — poster subgraph (interactive)
  kg_poster_static.png         — static poster image (300 DPI)
  kg_poster_static.svg         — static poster image (vector)
  T2DM_Melanoma_KG_Dataset_auto6.xlsx  — cleaned dataset (STRING 0.9)
  T2DM_Melanoma_KG_Results.xlsx        — all analysis results


In [27]:
# CELL 11 — Generate all publication-quality figures for the dissertation
#
# PURPOSE:
#   Produce five figures from the analysis results computed in Cells 5 and 6.
#   All figures are saved at 300 DPI for print-quality dissertation insertion.


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ── Shared style settings ─────────────────────────────────────────────────────
# All figures use the same dark background and colour scheme as the
# interactive graph for visual consistency throughout the dissertation.

# ── Shared style settings ─────────────────────────────────────────────────────
# Publication-friendly light theme

BG_COLOR    = "#ffffff"    # white background
TEXT_COLOR  = "#222222"    # dark text
GRID_COLOR  = "#d9d9d9"    # light grey grid
EDGE_COLOR  = "#666666"    # dark grey outlines
ACCENT      = "#e67e22"    # orange
SHARED_COL  = "#f1c40f"    # yellow
CROSS_COL   = "#1f2937"    # dark navy/charcoal (was white)
GENE_COL    = "#2e8b57"    # green
DRUG_BLUE   = "#377eb8"    # blue
ANNOT_BG    = "#f7f7f7"    # light grey annotation box

plt.rcParams.update({
    "figure.facecolor":  BG_COLOR,
    "axes.facecolor":    BG_COLOR,
    "axes.edgecolor":    EDGE_COLOR,
    "axes.labelcolor":   TEXT_COLOR,
    "xtick.color":       TEXT_COLOR,
    "ytick.color":       TEXT_COLOR,
    "text.color":        TEXT_COLOR,
    "grid.color":        GRID_COLOR,
    "grid.linewidth":    0.6,
    "font.family":       "Arial",
    "savefig.facecolor": BG_COLOR,
    "savefig.edgecolor": BG_COLOR,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

print("Generating all figures...")
print()

Generating all figures...



In [85]:
# FIGURE 1 — Betweenness centrality distribution — GENE NODES ONLY
#
# PURPOSE:
#   Shows the distribution of betweenness centrality across all 157 gene nodes.
#   Disease and pathway nodes are excluded because their high scores
#   are a structural artefact of the hub-and-spoke graph topology rather
#   than a biological finding.

fig1, ax1 = plt.subplots(figsize=(12, 6), facecolor="white")
ax1.set_facecolor("white")

# Gene nodes only
gene_bc_df = centrality_df[centrality_df["node_type"] == "Gene"].copy()
gene_bc    = gene_bc_df["betweenness"].values
akt2_bc    = gene_bc_df[gene_bc_df["name"] == "AKT2"]["betweenness"].values[0]
mean_bc    = gene_bc.mean()
med_bc     = np.median(gene_bc)

# Main histogram
ax1.hist(gene_bc, bins=35, color="#2d8a4e", edgecolor="white",
         linewidth=0.4, alpha=0.85)

# Reference lines
ax1.axvline(mean_bc, color="#e63946", lw=1.8, ls="--",
            label=f"Mean = {mean_bc:.4f}")
ax1.axvline(med_bc,  color="#e07b00", lw=1.8, ls=":",
            label=f"Median = {med_bc:.4f}")
ax1.axvline(akt2_bc, color="#1a1a6e", lw=2.2, ls="-",
            label=f"AKT2 = {akt2_bc:.4f}")

# Annotate key genes with arrows
key_genes = {
    "AKT2": akt2_bc,
    "TP53": gene_bc_df[gene_bc_df["name"] == "TP53"]["betweenness"].values[0],
    "INS":  gene_bc_df[gene_bc_df["name"] == "INS"]["betweenness"].values[0],
    "IRS1": gene_bc_df[gene_bc_df["name"] == "IRS1"]["betweenness"].values[0],
}

# Stagger annotation heights to prevent overlap
heights = {"AKT2": 22, "TP53": 17, "INS": 12, "IRS1": 7}
for gene, bc_val in key_genes.items():
    ax1.annotate(
        gene,
        xy         = (bc_val, 1),
        xytext     = (bc_val + 0.001, heights[gene]),
        fontsize   = 9.5,
        color      = "#1a1a6e" if gene == "AKT2" else "#333333",
        fontweight = "bold" if gene == "AKT2" else "normal",
        arrowprops = dict(arrowstyle="-", color="#aaaaaa", lw=0.8),
    )

ax1.set_xlabel("Betweenness Centrality Score", fontsize=12, color="#222222")
ax1.set_ylabel("Number of Gene Nodes", fontsize=12, color="#222222")
ax1.set_title(
    "Distribution of Betweenness Centrality Across 157 Gene Nodes",
    fontsize=13, color="#222222", pad=12
)
ax1.legend(fontsize=10, framealpha=0.9, edgecolor="#cccccc",
           facecolor="white", labelcolor="#222222", loc="upper right")
ax1.tick_params(colors="#222222")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["left"].set_color("#cccccc")
ax1.spines["bottom"].set_color("#cccccc")
ax1.grid(axis="y", alpha=0.3, color="#cccccc")

fig1.patch.set_facecolor("white")
plt.tight_layout()
plt.savefig("fig1_bc_distribution.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Saved: fig1_bc_distribution.png")

Saved: fig1_bc_distribution.png


In [60]:
# FIGURE 2 — Betweenness centrality: top 20 gene nodes

fig2, ax2 = plt.subplots(figsize=(12, 6))

# Get top 20 gene nodes
top20_genes = centrality_df[centrality_df["node_type"] == "Gene"].head(20).copy()
top20_genes = top20_genes.sort_values("betweenness", ascending=True)

# Assign colours — cross-disease=white, shared=yellow, other=green
bar_colors = []
for _, row in top20_genes.iterrows():
    nid = row["node_id"]
    if nid in cross_disease_genes:
        bar_colors.append(CROSS_COL)
    elif nid in shared_pathway_genes or "Shared" in str(row["notes"]):
        bar_colors.append(SHARED_COL)
    else:
        bar_colors.append(GENE_COL)

bars = ax2.barh(
    top20_genes["name"],
    top20_genes["betweenness"],
    color      = bar_colors,
    edgecolor  = EDGE_COLOR,
    linewidth  = 0.5,
    height     = 0.65,
)

# Add value labels on each bar
for bar, val in zip(bars, top20_genes["betweenness"]):
    ax2.text(
        bar.get_width() + 0.0005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va        = "center",
        fontsize  = 8.5,
        color     = TEXT_COLOR,
    )

ax2.set_xlabel("Betweenness Centrality Score (normalised)", fontsize=12)
ax2.set_title(
    " Betweenness Centrality of Top 20 Gene Nodes",
    fontsize=13, pad=12
)
ax2.grid(axis="x", alpha=0.4)

# Legend
legend_elements = [
    mpatches.Patch(color=CROSS_COL,  label="Cross-disease gene (AKT2)",
                   edgecolor=GRID_COLOR),
    mpatches.Patch(color=SHARED_COL, label="Shared pathway gene"),
    mpatches.Patch(color=GENE_COL,   label="Other gene"),
]
ax2.legend(handles=legend_elements, fontsize=10,
           framealpha=0.2, edgecolor=GRID_COLOR, loc="lower right")

plt.tight_layout()
plt.savefig("fig2_bc_top20_genes.png")
plt.close()
print("Saved: fig2_bc_top20_genes.png")


C:\Users\USER\AppData\Local\Temp\ipykernel_26160\769996244.py:49: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  mpatches.Patch(color=CROSS_COL,  label="Cross-disease gene (AKT2)",


Saved: fig2_bc_top20_genes.png


In [29]:
# # FIGURE 3 — Betweenness centrality vs degree scatter plot (all gene nodes)

# fig3, ax3 = plt.subplots(figsize=(11, 7))

# # Gene-only subset
# gene_centrality = centrality_df[centrality_df["node_type"] == "Gene"].copy()

# # Assign point colours
# point_colors = []
# point_sizes  = []
# point_zorder = []
# for _, row in gene_centrality.iterrows():
#     nid = row["node_id"]
#     if nid in cross_disease_genes:
#         point_colors.append(CROSS_COL)
#         point_sizes.append(200)
#         point_zorder.append(5)
#     elif nid in shared_pathway_genes or "Shared" in str(row["notes"]):
#         point_colors.append(SHARED_COL)
#         point_sizes.append(120)
#         point_zorder.append(4)
#     else:
#         point_colors.append(GENE_COL)
#         point_sizes.append(40)
#         point_zorder.append(3)

# # Plot each gene as a scatter point
# for i, (_, row) in enumerate(gene_centrality.iterrows()):
#     ax3.scatter(
#         row["degree"],
#         row["betweenness"],
#         c       = point_colors[i],
#         s       = point_sizes[i],
#         zorder  = point_zorder[i],
#         alpha   = 0.85,
#         edgecolors  = EDGE_COLOR,
#         linewidths = 0.4,
#     )

# # Label notable genes explicitly
# label_genes = {
#     "AKT2", "TP53", "INS", "IRS1", "EGFR",
#     "PIK3CA", "MAPK1", "MAPK3", "NRAS", "INSR"
# }
# for _, row in gene_centrality.iterrows():
#     if row["name"] in label_genes:
#         nid = row["node_id"]
#         if nid in cross_disease_genes:
#             col = CROSS_COL
#         elif nid in shared_pathway_genes or "Shared" in str(row["notes"]):
#             col = SHARED_COL
#         else:
#             col = TEXT_COLOR

#         ax3.annotate(
#             row["name"],
#             xy         = (row["degree"], row["betweenness"]),
#             xytext     = (6, 4),
#             textcoords = "offset points",
#             fontsize   = 8.5,
#             color      = col,
#             fontweight = "bold" if nid in cross_disease_genes else "normal",
#         )

# ax3.set_xlabel("Degree (number of connections)", fontsize=12)
# ax3.set_ylabel("Betweenness Centrality Score (normalised)", fontsize=12)
# ax3.set_title(
#     "Betweenness Centrality vs Degree for All 157 Gene Nodes",
#     fontsize=12, pad=12
# )
# ax3.grid(alpha=0.3)

# legend_elements = [
#     mpatches.Patch(color=CROSS_COL,  label="Cross-disease gene (AKT2)",
#                    edgecolor=GRID_COLOR),
#     mpatches.Patch(color=SHARED_COL, label="Shared pathway gene"),
#     mpatches.Patch(color=GENE_COL,   label="Other gene"),
# ]
# ax3.legend(handles=legend_elements, fontsize=10,
#            framealpha=0.2, edgecolor=GRID_COLOR)

# plt.tight_layout()
# plt.savefig("fig3_bc_vs_degree_scatter.png")
# plt.close()
# print("Saved: fig3_bc_vs_degree_scatter.png")



# FIGURE 3 — Betweenness centrality vs degree scatter plot
# Fixed: label offsets are now staggered per gene to prevent overlapping
# Font sizes increased substantially


fig3, ax3 = plt.subplots(figsize=(12, 8), facecolor="white")
ax3.set_facecolor("white")

gene_centrality = centrality_df[centrality_df["node_type"] == "Gene"].copy()

point_colors = []
point_sizes  = []
point_zorder = []
for _, row in gene_centrality.iterrows():
    nid = row["node_id"]
    if nid in cross_disease_genes:
        point_colors.append(CROSS_COL)
        point_sizes.append(250)
        point_zorder.append(5)
    elif nid in shared_pathway_genes or "Shared" in str(row["notes"]):
        point_colors.append(SHARED_COL)
        point_sizes.append(140)
        point_zorder.append(4)
    else:
        point_colors.append(GENE_COL)
        point_sizes.append(45)
        point_zorder.append(3)

for i, (_, row) in enumerate(gene_centrality.iterrows()):
    ax3.scatter(
        row["degree"], row["betweenness"],
        c          = point_colors[i],
        s          = point_sizes[i],
        zorder     = point_zorder[i],
        alpha      = 0.85,
        edgecolors = "#cccccc",
        linewidths = 0.4,
    )

# Staggered manual offsets per gene to prevent overlapping labels
# Each tuple is (x_offset, y_offset) in points
label_offsets = {
    "AKT2":   (8,   6),
    "TP53":   (8,  -12),
    "INS":    (-60, 8),
    "IRS1":   (8,   8),
    "EGFR":   (8,  -12),
    "PIK3CA": (8,   6),
    "MAPK1":  (-60, 6),
    "MAPK3":  (8,  -12),
    "NRAS":   (8,   6),
    "INSR":   (-60,-12),
}

label_genes = set(label_offsets.keys())

for _, row in gene_centrality.iterrows():
    if row["name"] in label_genes:
        nid = row["node_id"]
        col = (CROSS_COL if nid in cross_disease_genes
               else SHARED_COL if nid in shared_pathway_genes
               else TEXT_COLOR)
        xoff, yoff = label_offsets[row["name"]]
        ax3.annotate(
            row["name"],
            xy         = (row["degree"], row["betweenness"]),
            xytext     = (xoff, yoff),
            textcoords = "offset points",
            fontsize   = 10,         # increased from 8.5
            color      = col,
            fontweight = "bold" if nid in cross_disease_genes else "normal",
            arrowprops = dict(arrowstyle="-", color="#aaaaaa", lw=0.7)
            if abs(xoff) > 10 or abs(yoff) > 10 else None,
        )

ax3.set_xlabel("Degree (number of connections)", fontsize=12)
ax3.set_ylabel("Betweenness centrality score (normalised)", fontsize=12)
ax3.tick_params(axis="both", labelsize=11)
ax3.set_title(
    "Betweenness centrality versus degree for all 157 gene nodes",
    fontsize=13, pad=12, loc="left"
)
ax3.grid(alpha=0.3)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

legend_elements = [
    mpatches.Patch(color=CROSS_COL,  label="Cross-disease gene (AKT2)"),
    mpatches.Patch(color=SHARED_COL, label="Shared pathway gene"),
    mpatches.Patch(color=GENE_COL,   label="Other gene"),
]
ax3.legend(handles=legend_elements, fontsize=11,
           framealpha=0.9, edgecolor="#cccccc")

plt.tight_layout()
plt.savefig("fig3_bc_vs_degree_scatter.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Saved: fig3_bc_vs_degree_scatter.png")


Saved: fig3_bc_vs_degree_scatter.png


In [31]:
# # FIGURE 4 — Drug proximity by drug class (horizontal bar chart)

# fig4, ax4 = plt.subplots(figsize=(12, 7))

# # Sort drug classes by mean proximity ascending (closest at top)
# class_plot = class_summary.sort_values("mean_proximity", ascending=True).copy()

# # Colour bars: T2DM drug classes blue, melanoma-specific classes teal, other grey
# def class_color(name):
#     if "Insulin" in name:           return "#f6c90e"
#     if "GLP-1" in name:             return DRUG_BLUE
#     if "SGLT2" in name:             return "#63e6be"
#     if "DPP-4" in name:             return "#a78bfa"
#     if "Sulfonylurea" in name:      return "#ffa94d"
#     if "Biguanide" in name:         return "#ff4757"
#     if "Thiazolidinedione" in name: return "#da77f2"
#     if "BRAF" in name:              return "#4fd1c5"
#     if "Immunotherapy" in name:     return "#76e4f7"
#     if "Alpha" in name:             return "#f6ad55"
#     if "Amylin" in name:            return "#fc8181"
#     return "#aaaaaa"

# bar_colors_4 = [class_color(n) for n in class_plot["drug_class"]]

# bars4 = ax4.barh(
#     class_plot["drug_class"],
#     class_plot["mean_proximity"],
#     color      = bar_colors_4,
#     edgecolor  = EDGE_COLOR,
#     linewidth  = 0.5,
#     height     = 0.6,
# )

# # Add error bars showing min proximity (best individual drug in class)
# for i, (_, row) in enumerate(class_plot.iterrows()):
#     ax4.plot(
#         row["best_proximity"],
#         i,
#         marker    = "|",
#         color     = TEXT_COLOR,
#         markersize = 12,
#         markeredgewidth = 2,
#         alpha     = 0.7,
#         zorder    = 5,
#         label     = "Best individual drug" if i == 0 else "",
#     )

# # Value labels
# for bar, val in zip(bars4, class_plot["mean_proximity"]):
#     ax4.text(
#         bar.get_width() + 0.03,
#         bar.get_y() + bar.get_height() / 2,
#         f"{val:.3f}",
#         va       = "center",
#         fontsize = 9,
#         color    = TEXT_COLOR,
#     )

# ax4.set_xlabel("Mean Shortest Path to Melanoma Gene Set (lower = closer)", fontsize=11)
# ax4.set_title(
#     "Mean Network Proximity of Pharmacotherapy to Melanoma Biology",
#     fontsize=12, pad=12
# )
# ax4.grid(axis="x", alpha=0.4)
# ax4.legend(fontsize=9, framealpha=0.2, edgecolor=GRID_COLOR, loc="lower right")

# # # Add an annotation box explaining the score
# # ax4.text(
# #     0.98, 0.05,
# #     "Score = mean shortest path\nbetween drug targets and\nmelanoma pathway genes\n\nLower = mechanistically closer",
# #     transform    = ax4.transAxes,
# #     fontsize     = 8.5,
# #     color        = TEXT_COLOR,
# #     ha           = "right",
# #     va           = "bottom",
# #     bbox=dict(boxstyle="round,pad=0.4", facecolor=ANNOT_BG,
# #           edgecolor=EDGE_COLOR, alpha=1.0)
# # )

# plt.tight_layout()
# plt.savefig("fig4_drug_proximity_by_class.png")
# plt.close()
# print("Saved: fig4_drug_proximity_by_class.png")



# FIGURE 4 — Drug proximity by drug class
# Font sizes increased
# Axis label updated with structural-only disclaimer

fig4, ax4 = plt.subplots(figsize=(13, 7), facecolor="white")
ax4.set_facecolor("white")

class_plot = class_summary.sort_values("mean_proximity", ascending=True).copy()

def class_color(name):
    if "Insulin" in name:           return "#c9a800"
    if "GLP-1" in name:             return DRUG_BLUE
    if "SGLT2" in name:             return "#00897b"
    if "DPP-4" in name:             return "#7b2fa8"
    if "Sulfonylurea" in name:      return "#e07b00"
    if "Biguanide" in name:         return "#e63946"
    if "Thiazolidinedione" in name: return "#9b59b6"
    if "BRAF" in name:              return "#4fd1c5"
    if "Immunotherapy" in name:     return "#76e4f7"
    if "Alpha" in name:             return "#f6ad55"
    if "Amylin" in name:            return "#fc8181"
    return "#aaaaaa"

bar_colors_4 = [class_color(n) for n in class_plot["drug_class"]]

bars4 = ax4.barh(
    class_plot["drug_class"],
    class_plot["mean_proximity"],
    color     = bar_colors_4,
    edgecolor = "#cccccc",
    linewidth = 0.5,
    height    = 0.6,
)

for i, (_, row) in enumerate(class_plot.iterrows()):
    ax4.plot(
        row["best_proximity"], i,
        marker          = "|",
        color           = "#222222",
        markersize      = 14,
        markeredgewidth = 2.0,
        alpha           = 0.8,
        zorder          = 5,
        label           = "Best individual drug" if i == 0 else "",
    )

for bar, val in zip(bars4, class_plot["mean_proximity"]):
    ax4.text(
        bar.get_width() + 0.03,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va       = "center",
        fontsize = 10,     # increased from 9
        color    = TEXT_COLOR,
    )

ax4.set_xlabel(
    "Mean shortest path to melanoma gene set\n"
    "(lower = structurally closer; does not indicate therapeutic efficacy)",
    fontsize=11
)
ax4.tick_params(axis="both", labelsize=11)
ax4.set_title(
    "Mean Network Proximity of Pharmacotherapy to Melanoma Biology",
    fontsize=13, pad=12, loc="left"
)
ax4.grid(axis="x", alpha=0.4)
ax4.legend(fontsize=10, framealpha=0.9, edgecolor="#cccccc", loc="lower right")
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("fig4_drug_proximity_by_class.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Saved: fig4_drug_proximity_by_class.png")

Saved: fig4_drug_proximity_by_class.png


In [34]:
# FIGURE 4a — Mean network proximity to melanoma biology by T2DM drug class for poster
#
# PURPOSE:
#   Horizontal bar chart ranking all T2DM drug classes by mean shortest path
#   length between drug target genes and the melanoma gene set (78 genes).
#   Optimised for poster visibility — large fonts, thick bars, bold labels.
#
# POSTER OPTIMISATION:
#   - Figure size increased to (16, 9) for poster scale
#   - All font sizes substantially increased for readability at distance
#   - Bar height increased for visual weight on a poster panel
#   - Value labels enlarged and bold
#   - Best-drug marker enlarged
#   - Saved at 300 DPI for print quality


fig4, ax4 = plt.subplots(figsize=(16, 9), facecolor="white")
ax4.set_facecolor("white")

# ── Step 1: Sort drug classes by mean proximity ascending ─────────────────────
# Closest drug class to melanoma biology appears at the top of the chart.
# ascending=True for barh means top bar = lowest score = closest to melanoma.

class_plot = class_summary.sort_values("mean_proximity", ascending=True).copy()


# ── Step 2: Assign a distinct colour to each drug class ───────────────────────
# Colours are chosen to be visually distinct from each other on a white
# background and consistent with the colour scheme used in the full graph.

def class_color(name):
    """Return a distinct colour for each pharmacological drug class."""
    if "Insulin" in name:           return "#c9a800"   # dark yellow
    if "GLP-1" in name:             return DRUG_BLUE   # dark blue
    if "SGLT2" in name:             return "#00897b"   # teal
    if "DPP-4" in name:             return "#7b2fa8"   # purple
    if "Sulfonylurea" in name:      return "#e07b00"   # dark orange
    if "Biguanide" in name:         return "#e63946"   # dark red
    if "Thiazolidinedione" in name: return "#9b59b6"   # medium purple
    if "BRAF" in name:              return "#4fd1c5"   # teal-green
    if "Immunotherapy" in name:     return "#76e4f7"   # light blue
    if "Alpha" in name:             return "#f6ad55"   # light orange
    if "Amylin" in name:            return "#fc8181"   # light red
    return "#aaaaaa"                                   # grey fallback

bar_colors_4 = [class_color(n) for n in class_plot["drug_class"]]


# ── Step 3: Draw horizontal bars ──────────────────────────────────────────────
# Each bar represents one drug class; bar length = mean proximity score.
# Lower score = shorter bar = closer to melanoma biology in the network.

bars4 = ax4.barh(
    class_plot["drug_class"],
    class_plot["mean_proximity"],
    color     = bar_colors_4,
    edgecolor = "#cccccc",
    linewidth = 0.8,
    height    = 0.75,           # thicker bars for poster visibility
)


# ── Step 4: Add best-individual-drug markers ───────────────────────────────────
# A vertical tick mark (|) shows the best proximity score of any single drug
# within each class, indicating within-class variation.

for i, (_, row) in enumerate(class_plot.iterrows()):
    ax4.plot(
        row["best_proximity"], i,
        marker          = "|",
        color           = "#111111",
        markersize      = 22,       # substantially larger for poster visibility
        markeredgewidth = 3.0,      # thicker marker line
        alpha           = 0.9,
        zorder          = 5,
        label           = "Best individual drug" if i == 0 else "",
    )


# ── Step 5: Add value labels on each bar ──────────────────────────────────────
# Numeric score shown just to the right of each bar end.

for bar, val in zip(bars4, class_plot["mean_proximity"]):
    ax4.text(
        bar.get_width() + 0.04,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va         = "center",
        fontsize   = 14,        # large for poster visibility
        fontweight = "bold",
        color      = "#222222",
    )


# ── Step 6: Axis formatting ────────────────────────────────────────────────────
# Disclaimer included in axis label that proximity = structural distance only.

ax4.set_xlabel(
    "Mean shortest path to melanoma gene set\n"
    "(lower = structurally closer; does not indicate therapeutic efficacy)",
    fontsize = 15,         # large axis label for poster
    labelpad = 10,
)

ax4.tick_params(axis="both", labelsize=14)   # large tick labels for poster

ax4.set_title(
    "Mean Network Proximity of Pharmacotherapy to Melanoma Biology",
    fontsize = 17,         # large title for poster
    pad      = 14,
    loc      = "left",
)

ax4.grid(axis="x", alpha=0.35)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)


# ── Step 7: Legend ────────────────────────────────────────────────────────────
ax4.legend(
    fontsize   = 13,
    framealpha = 0.9,
    edgecolor  = "#cccccc",
    facecolor  = "white",
    loc        = "lower right",
)


# ── Step 8: Save ──────────────────────────────────────────────────────────────
plt.tight_layout()
plt.savefig("fig4_drug_proximity_by_class_poster.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Saved: fig4_drug_proximity_by_class_poster.png (poster-optimised, 300 DPI)")

Saved: fig4_drug_proximity_by_class_poster.png (poster-optimised, 300 DPI)


In [36]:
# FIGURE 5 — Drug proximity heatmap (individual drugs within each class)

# Prepare data: individual scored drugs with class labels
# Use scored_df which already has drug_class column
heatmap_data = scored_df[["drug_name", "drug_class", "mean_proximity"]].copy()

# Shorten very long drug names for readability
heatmap_data["drug_short"] = heatmap_data["drug_name"].apply(
    lambda x: x[:35] + "…" if len(x) > 35 else x
)

# Sort by drug class then proximity within class
heatmap_data = heatmap_data.sort_values(
    ["drug_class", "mean_proximity"]
).reset_index(drop=True)

# Create pivot-style data for heatmap
# Group drugs into their classes for the y-axis ordering
classes_ordered = class_summary.sort_values("mean_proximity")["drug_class"].tolist()
heatmap_data["class_order"] = heatmap_data["drug_class"].apply(
    lambda x: classes_ordered.index(x) if x in classes_ordered else 99
)
heatmap_data = heatmap_data.sort_values(["class_order", "mean_proximity"])

fig5, ax5 = plt.subplots(figsize=(10, 14))

# Colour scale: green (low/close) → red (high/distant)
cmap = mcolors.LinearSegmentedColormap.from_list(
    "proximity",
    ["#68d391", "#f6c90e", "#ff4757"],   # green → yellow → red
    N=256
)

# Normalize scores for colour mapping
min_score = heatmap_data["mean_proximity"].min()
max_score = heatmap_data["mean_proximity"].max()
norm      = mcolors.Normalize(vmin=min_score, vmax=max_score)

# Draw one horizontal bar per drug
y_positions  = range(len(heatmap_data))
bar_heights  = 0.7

for i, (_, row) in enumerate(heatmap_data.iterrows()):
    color = cmap(norm(row["mean_proximity"]))
    ax5.barh(
        i,
        row["mean_proximity"] - min_score + 0.1,   # offset so shortest bar still visible
        left       = min_score - 0.1,
        color      = color,
        edgecolor  = EDGE_COLOR,
        linewidth  = 0.3,
        height     = bar_heights,
    )
    # Drug name label on left
    ax5.text(
        min_score - 0.15,
        i,
        row["drug_short"],
        ha       = "right",
        va       = "center",
        fontsize = 6.5,
        color    = TEXT_COLOR,
    )
    # Score label on right
    ax5.text(
        row["mean_proximity"] + 0.03,
        i,
        f"{row['mean_proximity']:.3f}",
        ha       = "left",
        va       = "center",
        fontsize = 6.5,
        color    = TEXT_COLOR,
    )

# Add class separator lines and class labels
current_class  = None
class_start    = 0
class_label_y  = []

for i, (_, row) in enumerate(heatmap_data.iterrows()):
    if row["drug_class"] != current_class:
        if current_class is not None:
            ax5.axhline(i - 0.5, color=GRID_COLOR, linewidth=1.0, alpha=0.8)
            class_label_y.append((class_start, i - 1, current_class))
        current_class = row["drug_class"]
        class_start   = i

# Last class
class_label_y.append((class_start, len(heatmap_data) - 1, current_class))

# Add class label annotations on the right side
for start, end, label in class_label_y:
    mid = (start + end) / 2
    ax5.text(
        max_score + 0.2,
        mid,
        label,
        ha       = "left",
        va       = "center",
        fontsize = 7.5,
        color    = ACCENT,
        fontweight = "bold",
    )

# Colourbar
sm  = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig5.colorbar(sm, ax=ax5, orientation="horizontal",
                     fraction=0.02, pad=0.02, shrink=0.6)
cbar.set_label("Mean Proximity Score (lower = closer to melanoma)",
               color=TEXT_COLOR, fontsize=9)
cbar.ax.xaxis.set_tick_params(color=TEXT_COLOR)
plt.setp(cbar.ax.xaxis.get_ticklabels(), color=TEXT_COLOR, fontsize=8)

ax5.set_xlim(min_score - 1.5, max_score + 1.8)
ax5.set_ylim(-0.5, len(heatmap_data) - 0.5)
ax5.set_yticks([])
ax5.set_xlabel("Mean Network Proximity Score", fontsize=10)
ax5.set_title(
    "Network Proximity to Melanoma Biology for All 63 Scored Drugs\n"
    "(Grouped and ordered by pharmacological class)",
    fontsize=12, pad=12
)

plt.tight_layout()
plt.savefig("fig5_drug_proximity_heatmap.png")
plt.close()
print("Saved: fig5_drug_proximity_heatmap.png")


Saved: fig5_drug_proximity_heatmap.png


In [39]:
# FIGURE 5a — Network proximity to melanoma biology for all 63 scored drugs for poster
#
# PURPOSE:
#   Heatmap showing individual drug proximity scores for all 63 scored T2DM
#   drugs, ordered by pharmacological class and coloured by score value.
#   Green = closest to melanoma biology; red = most distant.
#   Optimised for poster visibility — larger fonts throughout.
#
# POSTER OPTIMISATION:
#   - Figure size increased to (14, 20) for poster scale
#   - Drug name font increased to 10pt
#   - Score label font increased to 10pt
#   - Class label font increased to 12pt and bold
#   - Colourbar label increased to 11pt
#   - Saved at 300 DPI
#
# PLACEMENT: Poster Results section alongside Fig 4 and the subgraph

# ── Step 1: Prepare heatmap data ──────────────────────────────────────────────
# Shorten drug names that are too long to display cleanly on the poster.
# Sort drugs by class (ordered by class mean proximity) then by individual
# proximity within each class so the best drug appears at the top of each group.

heatmap_data = scored_df[["drug_name", "drug_class", "mean_proximity"]].copy()

heatmap_data["drug_short"] = heatmap_data["drug_name"].apply(
    lambda x: x[:30] + "…" if len(x) > 30 else x   # truncate at 30 chars
)

# Order drug classes from closest to most distant (matches Fig 4 ordering)
classes_ordered = class_summary.sort_values("mean_proximity")["drug_class"].tolist()
heatmap_data["class_order"] = heatmap_data["drug_class"].apply(
    lambda x: classes_ordered.index(x) if x in classes_ordered else 99
)
heatmap_data = heatmap_data.sort_values(
    ["class_order", "mean_proximity"]
).reset_index(drop=True)


# ── Step 2: Set up colour map ─────────────────────────────────────────────────
# Three-colour gradient: green (low/close) → yellow (mid) → red (high/distant)
# This provides clear visual distinction across the full score range.

fig5, ax5 = plt.subplots(figsize=(14, 20), facecolor="white")
ax5.set_facecolor("white")

cmap = mcolors.LinearSegmentedColormap.from_list(
    "proximity",
    ["#2d8a4e", "#c9a800", "#e63946"],   # green → yellow → red
    N=256
)
min_score = heatmap_data["mean_proximity"].min()
max_score = heatmap_data["mean_proximity"].max()
norm      = mcolors.Normalize(vmin=min_score, vmax=max_score)


# ── Step 3: Identify class boundaries for separator lines and labels ──────────
# Track where each drug class starts and ends so we can draw a horizontal
# line between classes and add a class label on the right side.

current_class = None
class_start   = 0
class_label_y = []   # list of (start_index, end_index, class_name) tuples

for i, (_, row) in enumerate(heatmap_data.iterrows()):
    if row["drug_class"] != current_class:
        if current_class is not None:
            # Draw separator line between classes
            ax5.axhline(i - 0.5, color="#cccccc", linewidth=1.2)
            class_label_y.append((class_start, i - 1, current_class))
        current_class = row["drug_class"]
        class_start   = i

# Append the final class
class_label_y.append((class_start, len(heatmap_data) - 1, current_class))


# ── Step 4: Draw one bar per drug ─────────────────────────────────────────────
# Each bar is coloured by its proximity score using the green-yellow-red
# colour map. Drug name appears to the left; numeric score to the right.

for i, (_, row) in enumerate(heatmap_data.iterrows()):
    color = cmap(norm(row["mean_proximity"]))   # map score to colour

    ax5.barh(
        i,
        row["mean_proximity"] - min_score + 0.1,   # offset so shortest bar is visible
        left      = min_score - 0.1,
        color     = color,
        edgecolor = "white",
        linewidth = 0.3,
        height    = 0.75,          # slightly taller bars for poster visibility
    )

    # Drug name label — left of bar
    ax5.text(
        min_score - 0.18, i,
        row["drug_short"],
        ha       = "right",
        va       = "center",
        fontsize = 10,             # increased from 8 for poster visibility
        color    = "#222222",
    )

    # Numeric score label — right of bar
    ax5.text(
        row["mean_proximity"] + 0.06, i,
        f"{row['mean_proximity']:.3f}",
        ha       = "left",
        va       = "center",
        fontsize = 10,             # increased from 8 for poster visibility
        color    = "#222222",
    )


# ── Step 5: Add class labels on the right ─────────────────────────────────────
# Each class label is centred vertically within its group of drugs,
# positioned to the right of the score labels.

for start, end, label in class_label_y:
    mid = (start + end) / 2
    ax5.text(
        max_score + 0.35, mid,
        label,
        ha         = "left",
        va         = "center",
        fontsize   = 12,           # increased from 9 for poster visibility
        color      = "#e07b00",    # dark orange — visually prominent
        fontweight = "bold",
    )


# ── Step 6: Colourbar ─────────────────────────────────────────────────────────
# Horizontal colourbar below the chart shows the score-to-colour mapping
# and reinforces the proximity disclaimer.

sm   = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig5.colorbar(sm, ax=ax5, orientation="horizontal",
                     fraction=0.02, pad=0.02, shrink=0.55)
cbar.set_label(
     "Mean Proximity Score (lower = closer to melanoma; not therapeutic efficacy)",
    color    = "#222222",
    fontsize = 11,         # increased for poster visibility
)
plt.setp(cbar.ax.xaxis.get_ticklabels(), color="#222222", fontsize=10)


# ── Step 7: Axis formatting ────────────────────────────────────────────────────
ax5.set_xlim(min_score - 2.2, max_score + 2.8)
ax5.set_ylim(-0.5, len(heatmap_data) - 0.5)
ax5.set_yticks([])   # y-axis ticks suppressed — drug names serve as labels

ax5.set_xlabel(
    "Mean network proximity score",
    fontsize = 14,         # large axis label for poster
    labelpad = 8,
)

ax5.set_title(
    "Network Proximity to Melanoma Biology for All 63 Scored Drugs\n"
    "(Grouped and ordered by pharmacological class)",
    fontsize = 17,         # large title for poster
    pad      = 14,
)


# ── Step 8: Save ──────────────────────────────────────────────────────────────
plt.tight_layout()
plt.savefig("fig5_drug_proximity_heatmap_poster.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Saved: fig5_drug_proximity_heatmap_poster.png (poster-optimised, 300 DPI)")

Saved: fig5_drug_proximity_heatmap_poster.png (poster-optimised, 300 DPI)
